# Part 5: Plasticity dose tests (Tests 1 and 2)

**What this notebook does, in plain language.** A small network is asked again and again to memorise
fresh random labels for the *same* 64 MNIST images. Networks trained this way gradually get worse at
learning new labels ("loss of plasticity"). The claim under test is that this damage is driven by a
*dose*: every optimizer step moves the network's outputs by some amount ΔF, and steps that move the
outputs in directions the network finds hard (small eigenvalues of its tangent kernel K) count much more:

&nbsp;&nbsp;&nbsp;&nbsp;dose D = Σ_steps ΔFᵀ (K_ref + ρ λ_max I)⁻¹ ΔF.

The notebook checks three consequences: (Test 1) splitting a label change into N small steps divides the
dose and the damage by about N, and scaling a small label perturbation by ε scales them by ε²;
(Test 2) gradient descent filters its own jumps through K, an exact Gauss–Newton learner does not, so
optimizers should lose plasticity in the order of their dose; and (collapse) damage against cumulative dose
should fall on one curve for all optimizers. All predictions and pass/fail rules were fixed in advance.
**A FAIL is a valid result.** Nothing here is tuned after seeing damage numbers.

**How to run.**
- *Google Colab:* Runtime → Change runtime type → GPU (T4 is fine) → Runtime → Run all.
- *Any other Jupyter machine with a GPU (e.g. a rented RTX 5090):* run the cells down to and including the
  **Environment check** cell first. If it reports that the installed PyTorch build does not support this GPU (or that
  a library is missing), run the pip command it prints, restart the kernel, then Run all. Nothing is installed
  automatically. Outputs end up next to the notebook; download them from the Jupyter file browser.
- *Several GPUs (e.g. 4× RTX 3090):* nothing to configure. The notebook detects the GPUs and runs independent
  conditions in parallel worker processes (several per GPU); each condition gives exactly the numbers it would give
  on one GPU. The grid cell prints a progress line every minute (done / running / remaining, time left).
  **To stop**, interrupt that cell (■ / Kernel → Interrupt): the workers are terminated and finished conditions stay
  on disk. **To resume**, just Run all again: finished conditions are skipped, in parallel or sequential mode.

The default MODE is `"quick"` (1 seed, 8 tasks, a subset of conditions). Set MODE = `"full"` in the CONFIG
cell for the complete grid (3 seeds, 20 tasks). Results are saved to `results_part5.json` after every
condition; if the runtime disconnects, just Run all again and finished conditions are skipped.
To keep results across sessions, set `DRIVE_DIR` (Google Drive) in the CONFIG cell.

## 2. CONFIG
Everything you might want to change is here. Do not change the scientific constants after seeing results.

In [ ]:
# ============================== CONFIG ==============================
MODE = "quick"        # "quick": seed 0, M=8 tasks, conditions marked [q];  "full": seeds 0,1,2, M=20, all conditions
DRIVE_DIR = None      # e.g. "/content/drive/MyDrive/part5"  -> results, figures and zip are also copied there

# --- scientific constants (spec sections 1-4) ---
TAU = 1e-3            # "Fit": mean squared error (1/(nC))||r||^2 <= TAU
MICRO_TAU_RULE = "tau/N^2"  # spec amendment: intermediate micro-targets j<N fit at mse <= TAU/N^2; the last (j=N) at mse <= TAU
S_CAP = 4000          # max optimizer steps per micro-target (first-order learners)
GN_CAP = 30           # max Gauss-Newton iterations per micro-target
RHO = 1e-4            # dose / damage regulariser (primary)
RHO_HI = 1e-2         # second regulariser (D_K_hi, A_hi)
RHO_GN = 1e-6         # Gauss-Newton damping (relative to lambda_max)
GN_MAX_HALVINGS = 10  # step-halving line search on ||r||
N_TRAIN = 64          # n training images
WIDTH = 256           # hidden width of the MLPs
LIN_WIDTH = 2048      # frozen random-feature width of the linear control
N_CLASSES = 10        # C
PROBE_STEPS = 300     # probe trainability: plain GD steps on a clone
EIG_FLOOR = 1e-6      # directions with lambda_i > EIG_FLOOR*lambda_ref enter the filter exponent fit
LR_GRID_GD = [0.01, 0.03, 0.1, 0.3, 1.0]       # GD and MOM
LR_GRID_ADAM = [1e-4, 3e-4, 1e-3, 3e-3]        # ADAM (ADAM-R uses ADAM's value)
LR_PER_MODEL = True   # True: the same LR rule (sec. 3) is applied separately to the tanh arm and the linear control
                      # (the spec does not say which LR those models use; see the note in the LR-selection section)
TEST2_BLOCKING = False  # spec test 2 (one full GN step removes >= 90% of ||r||) is printed PASS/FAIL either way.
                        # False: a failure of test 2 alone does not stop the run IF the supplementary test 2b
                        # (GN implementation check) passes. True: any failing test stops the notebook.

# --- engineering ---
JAC_CHUNK = 128       # Jacobian rows per vmap chunk (lower it if the GPU runs out of memory; raising it changes speed only)
GRAM_CHUNK = 32768    # parameter columns per float64 chunk when forming K = J J^T
RESULTS_FILE = "results_part5.json"
RUN_DIR = "part5_run"   # one result file per finished condition (+ claims, queue and worker logs for parallel runs)
FIG_DIR = "figures_part5"
ZIP_FILE = "part5_outputs.zip"

# --- parallel execution (changes speed only, never the numbers) ---
PARALLEL = True           # True: worker processes when >= 2 GPUs are found; False: always the sequential single-process path
NUM_GPUS = "auto"         # "auto" = torch.cuda.device_count(); 0 or 1 -> sequential single-process run (as before)
WORKERS_PER_GPU = "auto"  # "auto" = min(3, usable CPU cores // GPUs), at least 1; reduced if GPU memory would be short
THREADS_PER_WORKER = 2    # CPU threads per worker process
CPU_WORKERS = 0           # testing only: > 0 runs this many worker processes on the CPU even without GPUs
PROGRESS_EVERY_S = 60     # seconds between progress lines while workers run

# --- grids (spec sec. 5); True = marked [q] (run in "quick") ---
GRIDS = [
    dict(name="G-main", model="relu",   seq="R", learners=["GD", "MOM", "ADAM", "ADAM-R"],
         Ns={1: True, 2: False, 4: True, 8: False, 16: True, 32: False}),
    dict(name="G-gn",   model="relu",   seq="R", learners=["GN"],
         Ns={1: True, 2: False, 4: True, 8: False, 16: True}),
    dict(name="G-lr",   model="relu",   seq="R", learners=["GD-half", "GD-quarter"], Ns={1: True}),
    dict(name="G-eps",  model="relu",   seq="E", learners=["GN", "GD", "ADAM"], Ns={1: True},
         eps={0.05: False, 0.1: True, 0.2: False, 0.4: True}),
    dict(name="G-tanh", model="tanh",   seq="R", learners=["GD", "ADAM", "GN"], Ns={1: False, 8: False}),
    dict(name="G-lin",  model="linear", seq="R", learners=["GD", "ADAM", "GN"], Ns={1: True}),
]

# --- mode-dependent settings ---
if MODE == "full":
    SEEDS, M_TASKS, QUICK_ONLY, TINY_NS = [0, 1, 2], 20, False, None
elif MODE == "quick":
    SEEDS, M_TASKS, QUICK_ONLY, TINY_NS = [0], 8, True, None
elif MODE == "tiny":
    # local CPU validation only (spec sec. 9); not a science mode
    SEEDS, M_TASKS, QUICK_ONLY, TINY_NS = [0], 2, False, {1, 2}
    N_TRAIN, WIDTH, LIN_WIDTH, S_CAP = 16, 32, 128, 400
    PROGRESS_EVERY_S = 5
else:
    raise ValueError("MODE must be 'quick' or 'full'")
print(f"MODE={MODE}: seeds={SEEDS}, M={M_TASKS} tasks, n={N_TRAIN}, width={WIDTH}, S_cap={S_CAP}")

# everything the library (and every worker process) needs, as one dictionary
_CFG_NAMES = ["MODE", "DRIVE_DIR", "TAU", "MICRO_TAU_RULE", "S_CAP", "GN_CAP", "RHO", "RHO_HI", "RHO_GN",
              "GN_MAX_HALVINGS", "N_TRAIN", "WIDTH", "LIN_WIDTH", "N_CLASSES", "PROBE_STEPS", "EIG_FLOOR",
              "LR_GRID_GD", "LR_GRID_ADAM", "LR_PER_MODEL", "TEST2_BLOCKING", "JAC_CHUNK", "GRAM_CHUNK",
              "RESULTS_FILE", "RUN_DIR", "FIG_DIR", "ZIP_FILE", "THREADS_PER_WORKER", "GRIDS", "SEEDS", "M_TASKS",
              "QUICK_ONLY", "TINY_NS"]
CFG = {k: globals()[k] for k in _CFG_NAMES}

## 3. Library code (one source of truth)
The next cells (sections 3–10) do not compute anything: each one writes its part of the library file `part5lib.py`
(the first with `%%writefile`, the rest with `%%writefile -a`, i.e. append). The notebook kernel and every parallel
worker process import this one file, so the science code exists exactly once. To change the code, edit these cells
and run them again (always from this first one, which starts the file afresh).
Section 3 holds the imports, configuration, device settings and the environment check.

In [ ]:
%%writefile part5lib.py
"""part5lib - all library code of the Part 5 plasticity-dose notebook (data, models, kernel utilities, learners,
measurement, runner, parallel workers). Written by the notebook's %%writefile cells; imported by the notebook kernel
and by every worker process. Do not edit by hand: edit the notebook cells and re-run them."""
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS (must precede CUDA init)
import sys, re, json, time, math, copy, shutil, zipfile, glob, gc, warnings, platform, traceback, socket, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.func import functional_call, vjp, vmap
try:
    from scipy import stats
    HAVE_SCIPY = True
except Exception as _e:
    stats = None
    HAVE_SCIPY = False

try:
    import torchvision
    HAVE_TORCHVISION = True
except Exception as _e:
    HAVE_TORCHVISION = False
try:
    import matplotlib          # only checked here; the figures cell imports pyplot itself
    HAVE_MPL = True
except Exception as _e:
    HAVE_MPL = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CFG = {}

def configure(cfg):
    """Install the CONFIG-cell settings as module globals (kernel: from CFG; workers: from config.json)."""
    CFG.clear()
    CFG.update(cfg)
    globals().update(cfg)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_device(dev):
    global DEVICE
    DEVICE = torch.device(dev)
torch.backends.cuda.matmul.allow_tf32 = False       # TF32 would corrupt small kernel eigenvalues
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
torch.use_deterministic_algorithms(True, warn_only=True)

def gen(seed):
    """A CPU torch.Generator with a fixed seed (all spec seeds use this)."""
    return torch.Generator().manual_seed(int(seed))

def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def fmt_time(s):
    s = float(s)
    if s < 120: return f"{s:.0f}s"
    if s < 7200: return f"{s/60:.1f} min"
    return f"{s/3600:.1f} h"

def mirror(path):
    """Copy a file to DRIVE_DIR if set."""
    if CFG.get("DRIVE_DIR") and os.path.exists(path):
        try:
            shutil.copy(path, os.path.join(DRIVE_DIR, os.path.basename(path)))
        except Exception as e:
            print("copy to DRIVE_DIR failed:", e)

# ---------------- environment check ----------------
def env_check():
    """Inspect and print only (never installs). Falls back to the CPU if the GPU is unusable."""
    global DEVICE
    print(f"python {platform.python_version()} | torch {torch.__version__} | torch.version.cuda = {torch.version.cuda}")
    with warnings.catch_warnings(record=True) as wlist:
        warnings.simplefilter("always")
        cuda_ok = torch.cuda.is_available()
        info = {}
        if cuda_ok:
            try:
                props = torch.cuda.get_device_properties(0)
                cap = torch.cuda.get_device_capability(0)
                info = dict(name=props.name, mem_gb=props.total_memory / 1024 ** 3, cap=cap,
                            arch_list=torch.cuda.get_arch_list())
            except Exception as e:
                info = dict(error=repr(e))
    print(f"CUDA available: {cuda_ok}")
    if info:
        if "error" in info:
            print("  could not query the GPU:", info["error"])
        else:
            print(f"  GPU: {info['name']} | total memory {info['mem_gb']:.1f} GB | compute capability "
                  f"sm_{info['cap'][0]}{info['cap'][1]} | this build has kernels for: {' '.join(info['arch_list'])}")
    print(f"libraries: torchvision={'yes' if HAVE_TORCHVISION else 'NO'} | scipy={'yes' if HAVE_SCIPY else 'NO'} | "
          f"matplotlib={'yes' if HAVE_MPL else 'NO'}")

    problem = None
    if not cuda_ok:
        problem = "CUDA is not available to this PyTorch build (torch.cuda.is_available() is False)."
    else:
        if "cap" in info:
            sm = f"sm_{info['cap'][0]}{info['cap'][1]}"
            if sm not in info["arch_list"]:
                problem = (f"this PyTorch build has no compiled kernels for {sm} "
                           f"(it has: {' '.join(info['arch_list'])}).")
        try:
            with warnings.catch_warnings(record=True) as w2:
                warnings.simplefilter("always")
                dev = torch.device("cuda")
                a32 = torch.randn(64, 64, device=dev)
                a64 = torch.randn(64, 64, device=dev, dtype=torch.float64)
                b = (a32 @ a32.T).sum().item() + (a64 @ a64.T).sum().item()
                ev = torch.linalg.eigh(a64 @ a64.T + torch.eye(64, device=dev, dtype=torch.float64))[0]
                net = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 3)).to(dev)
                prm = {k: v.detach() for k, v in net.named_parameters()}
                xb = torch.randn(5, 8, device=dev)
                jac = vmap(torch.func.jacrev(lambda pp, x: functional_call(net, pp, (x,))), in_dims=(None, 0))(prm, xb)
                torch.cuda.synchronize()
                assert math.isfinite(b) and bool(torch.isfinite(ev).all()) and jac["0.weight"].shape == (5, 3, 16, 8)
            wlist.extend(w2)
            print("GPU smoke test (float32/float64 matmul, float64 eigh, torch.func vmap/jacrev): OK")
        except Exception as e:
            problem = f"the GPU smoke test failed: {e!r}"
    bad_w = [str(w.message) for w in wlist if any(k in str(w.message).lower() for k in
             ("capability", "not compatible", "sm_", "no kernel image"))]
    if bad_w and problem is None:
        problem = "PyTorch warned that this GPU is not supported: " + " | ".join(bad_w)

    if problem is not None:
        if cuda_ok:
            print("\n" + "#" * 100 + "\n#### THE INSTALLED PYTORCH BUILD DOES NOT SUPPORT THIS GPU\n#### " + problem +
                  "\n#### Fix: run this in a terminal (or in a notebook cell with a leading '!'), then RESTART THE KERNEL:\n"
                  "####     pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu128\n"
                  "####   and if plotting or scipy is missing:\n####     pip install matplotlib scipy\n"
                  "#### Continuing on the CPU for now (correct but very slow).\n" + "#" * 100)
        else:
            print("\n" + "#" * 100 + "\n#### WARNING: NO USABLE GPU. " + problem +
                  "\n#### On Colab: Runtime -> Change runtime type -> GPU. On another machine with a GPU, install a CUDA build:\n"
                  "####     pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu128\n"
                  "####   (then restart the kernel). Continuing on the CPU (correct but very slow).\n" + "#" * 100)
        DEVICE = torch.device("cpu")
    if not (HAVE_TORCHVISION and HAVE_SCIPY and HAVE_MPL):
        print("Missing libraries: run   pip install matplotlib scipy   (and torchvision via the command above if needed),\n"
              "then restart the kernel. Without them: no MNIST download (synthetic fallback), no figures.")
    print(f"==> computations will run on: {DEVICE}")

## 4. Data
The first n images of a fixed per-seed permutation of MNIST-train, average-pooled 2×2 to 14×14 (196 inputs),
scaled to [0,1], then standardised with the scalar mean and std of those n images.
Targets: Y₀ = one-hot true labels; sequence R = fresh uniform random labels per task;
sequence E(ε) = Y₀ + ε·G on odd tasks and Y₀ on even tasks. A task is split into N micro-targets.

In [ ]:
%%writefile -a part5lib.py

# ================= 4. data =================
_MNIST_RAW = None
DATA_SOURCE = None

def _load_mnist_raw():
    global _MNIST_RAW, DATA_SOURCE
    if _MNIST_RAW is not None:
        return _MNIST_RAW
    force = CFG.get("FORCE_DATA_SOURCE")        # set for workers: they use what the kernel already loaded
    if HAVE_TORCHVISION and force != "synthetic":
        try:
            ds = torchvision.datasets.MNIST(root=CFG.get("MNIST_ROOT", "./mnist_data"), train=True,
                                            download=CFG.get("MNIST_DOWNLOAD", True))
            _MNIST_RAW = (ds.data.clone(), ds.targets.clone())
            DATA_SOURCE = "mnist"
        except Exception as e:
            print("MNIST download failed:", repr(e))
    if _MNIST_RAW is None:
        if force == "mnist":
            raise RuntimeError("worker could not read the MNIST files that the notebook kernel downloaded")
        _MNIST_RAW = "synthetic"
        DATA_SOURCE = "synthetic"
        print("\n" + "!" * 78 + "\n!!! WARNING: MNIST COULD NOT BE LOADED. FALLING BACK TO SYNTHETIC N(0,1) INPUTS !!!\n"
              "!!! with random labels. Results are NOT MNIST results. data_source='synthetic' !!!\n" + "!" * 78 + "\n")
    return _MNIST_RAW

_DATA_CACHE = {}

def get_data(seed):
    """Returns X (n,196) float32, labels (n,), Y0 (n,C) one-hot, all on DEVICE."""
    if seed in _DATA_CACHE:
        return _DATA_CACHE[seed]
    raw = _load_mnist_raw()
    g = gen(1000 + seed)
    if raw == "synthetic":
        x = torch.randn(N_TRAIN, 196, generator=g)
        labels = torch.randint(0, N_CLASSES, (N_TRAIN,), generator=g)
    else:
        data, targets = raw
        perm = torch.randperm(data.shape[0], generator=g)
        idx = perm[:N_TRAIN]
        img = data[idx].float().div(255.0).unsqueeze(1)          # (n,1,28,28) in [0,1]
        x = Fnn.avg_pool2d(img, 2).reshape(N_TRAIN, 196)          # (n,196)
        labels = targets[idx].long()
    x = (x - x.mean()) / x.std(unbiased=False)                    # scalar mean/std of these n images
    Y0 = Fnn.one_hot(labels, N_CLASSES).float()
    out = (x.to(DEVICE), labels.to(DEVICE), Y0.to(DEVICE))
    _DATA_CACHE[seed] = out
    return out

def make_targets(seq, eps, seed, M, Y0):
    """List [Y_0, Y_1, ..., Y_M] of (n,C) float32 targets on DEVICE."""
    Ys = [Y0.clone()]
    if seq == "R":
        g = gen(2000 + seed)
        for m in range(1, M + 1):
            lab = torch.randint(0, N_CLASSES, (N_TRAIN,), generator=g)
            Ys.append(Fnn.one_hot(lab, N_CLASSES).float().to(DEVICE))
    elif seq == "E":
        g = gen(3000 + seed)
        for m in range(1, M + 1):
            if m % 2 == 1:
                G = torch.randn(N_TRAIN, N_CLASSES, generator=g).to(DEVICE)
                Ys.append(Y0 + eps * G)
            else:
                Ys.append(Y0.clone())
    else:
        raise ValueError(seq)
    return Ys

def probe_targets(seed):
    lab = torch.randint(0, N_CLASSES, (N_TRAIN,), generator=gen(4000 + seed))
    return Fnn.one_hot(lab, N_CLASSES).float().to(DEVICE)

def micro_tau(j, N):
    """Fit threshold for micro-target j of N (MICRO_TAU_RULE): TAU/N^2 for j < N, TAU for the last one (j = N)."""
    assert MICRO_TAU_RULE == "tau/N^2", MICRO_TAU_RULE
    return TAU if j == N else TAU / N ** 2

def micro_targets(Y_prev, Y_m, N):
    """Y_{m-1} + (j/N)(Y_m - Y_{m-1}), j=1..N; the last one is Y_m exactly."""
    out = []
    for j in range(1, N + 1):
        out.append(Y_m.clone() if j == N else Y_prev + (j / N) * (Y_m - Y_prev))
    return out

## 5. Models
Main: MLP 196→256→256→10 with ReLU (arm: tanh), PyTorch default init seeded by `torch.manual_seed(seed)`.
Control: frozen random ReLU features 196→2048, trainable linear readout 2048→10 only (its kernel cannot change).

In [ ]:
%%writefile -a part5lib.py

# ================= 5. models =================
class MLP(nn.Module):
    def __init__(self, act, width):
        super().__init__()
        self.act_name = act
        self.l1 = nn.Linear(196, width)
        self.l2 = nn.Linear(width, width)
        self.l3 = nn.Linear(width, N_CLASSES)

    def act(self, z):
        return torch.relu(z) if self.act_name == "relu" else torch.tanh(z)

    def forward(self, x, return_hidden=False):
        h1 = self.act(self.l1(x))
        h2 = self.act(self.l2(h1))
        out = self.l3(h2)
        return (out, [h1, h2]) if return_hidden else out


class LinearControl(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.act_name = "relu"
        self.feat = nn.Linear(196, width)
        self.readout = nn.Linear(width, N_CLASSES)
        self.feat.requires_grad_(False)       # frozen random features

    def forward(self, x, return_hidden=False):
        h = torch.relu(self.feat(x))
        out = self.readout(h)
        return (out, [h]) if return_hidden else out


def make_model(kind, seed):
    torch.manual_seed(seed)
    if kind == "relu":
        m = MLP("relu", WIDTH)
    elif kind == "tanh":
        m = MLP("tanh", WIDTH)
    elif kind == "linear":
        m = LinearControl(LIN_WIDTH)
    else:
        raise ValueError(kind)
    return m.to(DEVICE)

def trainable(model):
    return [p for p in model.parameters() if p.requires_grad]

def flat_params(model):
    return torch.cat([p.detach().reshape(-1) for p in trainable(model)])

@torch.no_grad()
def set_flat_params(model, vec):
    o = 0
    for p in trainable(model):
        k = p.numel()
        p.copy_(vec[o:o + k].view_as(p))
        o += k

def outputs(model, X):
    """F: flattened n*C output vector (row-major: index i*C + c)."""
    return model(X).reshape(-1)

## 6. Kernel and Jacobian utilities
J = ∂F/∂θ (nC × P) is built with `torch.func` (one forward, vmap over chunks of output cotangents).
K = J Jᵀ is formed in float64 (chunked over parameters) and eigen-decomposed in float64; eigenvalues are
sorted descending.

In [ ]:
%%writefile -a part5lib.py

# ================= 6. kernel and Jacobian =================
def jacobian(model, X, chunk=None):
    chunk = chunk or JAC_CHUNK
    names = [n for n, p in model.named_parameters() if p.requires_grad]
    params = tuple(p.detach() for n, p in model.named_parameters() if p.requires_grad)

    def f(*ps):
        return functional_call(model, dict(zip(names, ps)), (X,)).reshape(-1)

    Fv, vjp_fn = vjp(f, *params)
    nC = Fv.numel()
    P = sum(p.numel() for p in params)
    J = torch.empty(nC, P, dtype=Fv.dtype, device=Fv.device)
    eye = torch.eye(nC, dtype=Fv.dtype, device=Fv.device)
    for s in range(0, nC, chunk):
        E = eye[s:s + chunk]
        gs = vmap(vjp_fn)(E)
        J[s:s + E.shape[0]] = torch.cat([g.reshape(E.shape[0], -1) for g in gs], dim=1)
        del gs
    return J

def gram64(J):
    nC, P = J.shape
    K = torch.zeros(nC, nC, dtype=torch.float64, device=J.device)
    for s in range(0, P, GRAM_CHUNK):
        Jc = J[:, s:s + GRAM_CHUNK].double()
        K += Jc @ Jc.T
        del Jc
    return 0.5 * (K + K.T)

def jt_vec64(J, v):
    """J^T v in float64 (chunked)."""
    nC, P = J.shape
    out = torch.empty(P, dtype=torch.float64, device=J.device)
    for s in range(0, P, GRAM_CHUNK):
        out[s:s + GRAM_CHUNK] = J[:, s:s + GRAM_CHUNK].double().T @ v
    return out

def eig_desc(K):
    lam, Q = torch.linalg.eigh(K)
    return lam.flip(0), Q.flip(1)

def kernel_eig(model, X):
    J = jacobian(model, X)
    K = gram64(J)
    del J
    lam, Q = eig_desc(K)
    return dict(K=K, lam=lam, Q=Q)

## 7. Learners
GD, MOM (heavy ball 0.9), ADAM, ADAM-R (state reset each task) are full-batch on L = (1/(2nC))‖F−Y‖².
GN is damped Gauss–Newton (solve in float64) with a step-halving line search on ‖r‖.
Every optimizer step (and every GN iteration) reports ΔF = F_after − F_before to the dose accumulator.

In [ ]:
%%writefile -a part5lib.py

# ================= 7. learners =================
FIRST_ORDER = ("GD", "GD-half", "GD-quarter", "MOM", "ADAM", "ADAM-R")

def make_optimizer(learner, model, lr):
    ps = trainable(model)
    if learner in ("GD", "GD-half", "GD-quarter"):
        return torch.optim.SGD(ps, lr=lr, momentum=0.0)
    if learner == "MOM":
        return torch.optim.SGD(ps, lr=lr, momentum=0.9)
    if learner in ("ADAM", "ADAM-R"):
        return torch.optim.Adam(ps, lr=lr, betas=(0.9, 0.999), eps=1e-8)
    raise ValueError(learner)

def loss_fn(F, Yf):
    return 0.5 * (F - Yf).pow(2).mean()        # = (1/(2nC)) ||F - Y||^2

def mse_of(r):
    return r.double().pow(2).mean().item()      # (1/(nC)) ||r||^2

def train_first_order(model, X, Yf, opt, acc, cap, tau=None):
    """Train on one micro-target until mse <= tau (default TAU) or cap. Returns (steps, fit, diverged)."""
    tau = TAU if tau is None else tau
    steps, pending = 0, False
    F_prev = r_prev = th_prev = None
    while True:
        F = outputs(model, X)
        Fd = F.detach()
        if pending:
            acc.add(Fd - F_prev, r_prev, (flat_params(model) - th_prev).norm())
            pending = False
        r = Fd - Yf
        mse = mse_of(r)
        if not math.isfinite(mse) or mse > 1e8:
            return steps, False, True
        if mse <= tau:
            return steps, True, False
        if steps >= cap:
            return steps, False, False
        loss = loss_fn(F, Yf)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        th_prev, F_prev, r_prev = flat_params(model), Fd, r
        opt.step()
        steps += 1
        pending = True

def train_gn(model, X, Yf, acc, cap, tau=None):
    """Damped Gauss-Newton on one micro-target. Returns (iterations, fit, diverged, line_search_failed)."""
    tau = TAU if tau is None else tau
    it = 0
    while True:
        with torch.no_grad():
            F = outputs(model, X)
        r = F - Yf
        mse = mse_of(r)
        if not math.isfinite(mse):
            return it, False, True, False
        if mse <= tau:
            return it, True, False, False
        if it >= cap:
            return it, False, False, False
        J = jacobian(model, X)
        K = gram64(J)
        lmax = torch.linalg.eigvalsh(K)[-1]
        r64 = r.double()
        v = torch.linalg.solve(K + RHO_GN * lmax * torch.eye(K.shape[0], dtype=K.dtype, device=K.device), r64)
        dth = (-jt_vec64(J, v)).float()
        del J, K
        th0 = flat_params(model)
        rn0 = r64.norm()
        accepted = False
        with torch.no_grad():
            for h in range(GN_MAX_HALVINGS + 1):
                set_flat_params(model, th0 + (0.5 ** h) * dth)
                F_new = outputs(model, X)
                if (F_new - Yf).double().norm() < rn0:
                    accepted = True
                    break
        if not accepted:
            set_flat_params(model, th0)
            return it, False, False, True
        acc.add(F_new - F, r, (flat_params(model) - th0).norm())
        it += 1

## 8. Measurement utilities
**Per step (dose accumulator):** with K_ref = Σ λᵢ qᵢqᵢᵀ fixed for the whole task (computed at its start),
D_K += ΔFᵀ(K_ref + ρλ_ref I)⁻¹ΔF (ρ = 1e-4; also 1e-2), D_raw += ‖ΔF‖², W_move += ‖Δθ‖, band doses for the
TOP 10% / MID 80% / BOT 10% eigen-directions, and filter sums S_jump[i] += (qᵢ·ΔF)², S_res[i] += (qᵢ·r)².
**Per task boundary:** primary damage A = log tr[(K̂+ρI)⁻¹] − (value at m = 0) with K̂ = K·nC/tr K,
spectrum statistics, probe trainability ψ (300 GD steps on a clone at η_p = nC/λ_max) and its linearised
prediction ψ_lin, unit statistics, and the filter exponent α.

In [ ]:
%%writefile -a part5lib.py

# ================= 8. measurement =================
class DoseAcc:
    def __init__(self, eig):
        lam, Q = eig["lam"], eig["Q"]
        self.lam = lam
        self.QT = Q.T.contiguous()
        self.lref = float(lam[0].item())
        self.nC = lam.numel()
        self.k10 = int(round(0.1 * self.nC))
        self.wK = 1.0 / (lam + RHO * self.lref)
        self.wK_hi = 1.0 / (lam + RHO_HI * self.lref)
        z = lambda: torch.zeros((), dtype=torch.float64, device=lam.device)
        self.D_K, self.D_K_hi, self.D_raw, self.W_move = z(), z(), z(), z()
        self.D_top, self.D_mid, self.D_bot = z(), z(), z()
        self.S_jump = torch.zeros_like(lam)
        self.S_res = torch.zeros_like(lam)
        self.steps = 0
        self.first_dtheta = None
        self.first_dF = None
        self.last_dF = None

    def add(self, dF, r_prev, dtheta_norm):
        d = dF.double()
        p = self.QT @ d
        pr = self.QT @ r_prev.double()
        p2 = p * p
        p2w = p2 * self.wK
        k, nC = self.k10, self.nC
        self.D_K += p2w.sum()
        self.D_K_hi += (p2 * self.wK_hi).sum()
        self.D_raw += d @ d
        self.W_move += dtheta_norm.double()
        self.D_top += p2w[:k].sum()
        self.D_mid += p2w[k:nC - k].sum()
        self.D_bot += p2w[nC - k:].sum()
        self.S_jump += p2
        self.S_res += pr * pr
        if self.steps == 0:
            self.first_dtheta = float(dtheta_norm.item())
            self.first_dF = float(d.norm().item())
        self.last_dF = d
        self.steps += 1

    def totals(self):
        return dict(D_K=self.D_K.item(), D_K_hi=self.D_K_hi.item(), D_raw=self.D_raw.item(),
                    W_move=self.W_move.item(),
                    D_band=dict(TOP=self.D_top.item(), MID=self.D_mid.item(), BOT=self.D_bot.item()),
                    first_step_dtheta=self.first_dtheta, first_step_dF=self.first_dF)


def filter_alpha(lam, S_jump, S_res, lref):
    """OLS slope of log(S_jump/S_res) on log(lambda) over lambda_i > EIG_FLOOR*lambda_ref."""
    lam, Sj, Sr = (t.detach().double().cpu().numpy() for t in (lam, S_jump, S_res))
    m = (lam > EIG_FLOOR * lref) & (Sj > 0) & (Sr > 0)
    if m.sum() < 3:
        return None
    return float(np.polyfit(np.log(lam[m]), np.log(Sj[m] / Sr[m]), 1)[0])


def damage_raw(lam, rho):
    """log tr[(Khat + rho I)^-1], Khat = K * nC / tr(K)."""
    nC = lam.numel()
    lhat = lam * nC / lam.sum()
    return float(torch.log((1.0 / (lhat + rho)).sum()).item())


def probe(model, X, Yp, lam, Q):
    """Standardised probe: 300 plain-GD steps on a clone at eta_p = nC/lambda_max. Returns psi, psi_lin, L0."""
    nC = lam.numel()
    lmax = float(lam[0].item())
    eta_p = nC / lmax
    Ypf = Yp.reshape(-1)
    clone = copy.deepcopy(model)
    ps = trainable(clone)
    with torch.no_grad():
        r0 = outputs(clone, X) - Ypf
        L0 = loss_fn(r0 + Ypf, Ypf).item()
    for _ in range(PROBE_STEPS):
        L = loss_fn(outputs(clone, X), Ypf)
        grads = torch.autograd.grad(L, ps)
        with torch.no_grad():
            for p, g in zip(ps, grads):
                p -= eta_p * g
    with torch.no_grad():
        L_end = loss_fn(outputs(clone, X), Ypf).item()
    del clone
    r064 = r0.double()
    proj = Q.T @ r064
    fac = torch.clamp(1.0 - lam / lmax, min=0.0) ** (2 * PROBE_STEPS)
    psi_lin = float(((proj ** 2) * fac).sum().item() / (r064 @ r064).item())
    psi = L_end / L0 if L0 > 0 else None
    return psi, psi_lin, L0


@torch.no_grad()
def unit_stats(model, X):
    _, hs = model(X, return_hidden=True)
    dead = [float((h <= 0).all(dim=0).float().mean().item()) for h in hs]        # ReLU: never active
    sat = [float((h.abs() > 0.99).all(dim=0).float().mean().item()) for h in hs]  # tanh: saturated on every input
    H = hs[-1].double()
    norms = H.norm(dim=0)
    nz = norms > 1e-12
    if int(nz.sum()) >= 2:
        Hn = H[:, nz] / norms[nz]
        Cm = Hn.T @ Hn
        dup = int(torch.triu((Cm.abs() > 0.99), diagonal=1).sum().item())
    else:
        dup = 0
    fro2 = float((H ** 2).sum().item())
    s_max = float(torch.linalg.matrix_norm(H, ord=2).item()) if fro2 > 0 else 0.0
    srank = fro2 / s_max ** 2 if s_max > 0 else 0.0
    inactive = sat if getattr(model, "act_name", "relu") == "tanh" else dead
    return dict(dead_frac=dead, sat_frac=sat, inactive_frac=inactive, dup_pairs=dup, stable_rank=srank,
                theta_norm=float(flat_params(model).double().norm().item()))


def boundary_measure(model, X, Yp, eig):
    lam, Q = eig["lam"], eig["Q"]
    lam_np = lam.detach().cpu().numpy()
    lmax = float(lam_np[0])
    p = np.clip(lam_np, 0, None)
    p = p / p.sum()
    pz = p[p > 0]
    rec = dict(A_raw=damage_raw(lam, RHO), A_hi_raw=damage_raw(lam, RHO_HI),
               lam_max=lmax, trK=float(lam_np.sum()),
               lam_p10=float(np.percentile(lam_np, 10)), lam_p50=float(np.percentile(lam_np, 50)),
               lam_p90=float(np.percentile(lam_np, 90)),
               eff_rank=float(np.exp(-(pz * np.log(pz)).sum())),
               n_eig_above=int((lam_np > 1e-6 * lmax).sum()))
    psi, psi_lin, L0 = probe(model, X, Yp, lam, Q)
    rec.update(psi=psi, psi_lin=psi_lin, probe_L0=L0)
    rec.update(unit_stats(model, X))
    return rec

## 9. Runner (one condition)
One condition = (model, sequence, learner, N, seed). Task 0 fits Y₀ from the initial weights (as a single
target); tasks 1..M follow the sequence, each split into N micro-targets. K_ref for a task is the kernel at
the end of the previous task (for task 0: at initialisation). Cumulative quantities (`*_cum`) sum tasks 1..m.

**Fit thresholds (spec amendment, `MICRO_TAU_RULE = "tau/N^2"`):** an intermediate micro-target j < N counts as
fit at mse ≤ τ/N²; the last micro-target of every task (j = N) at mse ≤ τ, so the fit quality at every task boundary
is the same for all N (N = 1 is unchanged). Without this, a 1/N-sized label change (mse ≈ 0.18/N²) is already
below τ at N = 16 and about half of the micro-targets would be skipped with zero steps.
Step caps are unchanged. The thresholds used, zero-step micro-targets and UNFIT counts are recorded per task.

In [ ]:
%%writefile -a part5lib.py

# ================= 9. runner =================
def cond_key(c):
    e = f"|eps={c['eps']}" if c.get("eps") is not None else ""
    return f"{c['grid']}|{c['model']}|{c['seq']}{e}|{c['learner']}|N={c['N']}|seed={c['seed']}|M={c['M']}"

def all_conditions():
    out = []
    for g in GRIDS:
        eps_items = list(g["eps"].items()) if "eps" in g else [(None, True)]
        for seed in SEEDS:
            for learner in g["learners"]:
                for N, nq in g["Ns"].items():
                    for eps, eq in eps_items:
                        if TINY_NS is not None and N not in TINY_NS:
                            continue
                        if QUICK_ONLY and not (nq and eq):
                            continue
                        out.append(dict(grid=g["name"], model=g["model"], seq=g["seq"], eps=eps,
                                        learner=learner, N=N, seed=seed, M=M_TASKS))
    return out

def _clean(x):
    """JSON-safe: non-finite floats -> None."""
    if isinstance(x, dict):
        return {k: _clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_clean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        x = float(x)
        return x if math.isfinite(x) else None
    if isinstance(x, (np.integer,)):
        return int(x)
    return x

def train_task(model, X, micro, learner, opt, acc, s_cap, gn_cap):
    """Train through the micro-targets of one task (thresholds from micro_tau). Returns per-task counts."""
    N = len(micro)
    out = dict(steps=0, unfit=0, zero_step=0, ls_fail=0, diverged=False, micro_steps=[],
               tau_micro=micro_tau(1, N) if N > 1 else None, tau_last=micro_tau(N, N))
    for j, Yj in enumerate(micro, start=1):
        Yf, tau_j = Yj.reshape(-1), micro_tau(j, N)
        if learner == "GN":
            s, fit, div, lsf = train_gn(model, X, Yf, acc, gn_cap, tau=tau_j)
            out["ls_fail"] += int(lsf)
        else:
            s, fit, div = train_first_order(model, X, Yf, opt, acc, s_cap, tau=tau_j)
        out["steps"] += s
        out["unfit"] += int(not fit)
        out["zero_step"] += int(s == 0)
        out["micro_steps"].append(s)
        if div:
            out["diverged"] = True
            break
    return out

def run_condition(c, lr, M=None, s_cap=None, gn_cap=None, keep_filters=True):
    M = c["M"] if M is None else M
    s_cap = S_CAP if s_cap is None else s_cap
    gn_cap = GN_CAP if gn_cap is None else gn_cap
    t_start = time.time()
    seed, learner, N = c["seed"], c["learner"], c["N"]
    X, labels, Y0 = get_data(seed)
    Yp = probe_targets(seed)
    Ys = make_targets(c["seq"], c.get("eps"), seed, M, Y0)
    model = make_model(c["model"], seed)
    is_gn = (learner == "GN")
    opt = None if is_gn else make_optimizer(learner, model, lr)
    eig = kernel_eig(model, X)             # K_ref for task 0 (at initialisation)
    records, status = [], "done"
    A0 = A0_hi = None
    cum = dict(D_K=0.0, D_K_hi=0.0, D_raw=0.0, W_move=0.0)
    cum0 = dict(D_K=0.0, D_K_hi=0.0, D_raw=0.0, W_move=0.0)
    for m in range(M + 1):
        if learner == "ADAM-R" and m >= 1:
            opt = make_optimizer(learner, model, lr)          # reset optimizer state at every task
        acc = DoseAcc(eig)
        micro = [Ys[0]] if m == 0 else micro_targets(Ys[m - 1], Ys[m], N)
        tr = train_task(model, X, micro, learner, opt, acc, s_cap, gn_cap)
        diverged = tr["diverged"]
        tot = acc.totals()
        rec = dict(m=m, steps=tr["steps"], unfit=tr["unfit"], n_micro=len(micro), zero_step_micro=tr["zero_step"],
                   gn_line_search_fail=tr["ls_fail"], tau_micro=tr["tau_micro"], tau_last=tr["tau_last"],
                   micro_steps=tr["micro_steps"], **tot)
        rec["alpha"] = filter_alpha(acc.lam, acc.S_jump, acc.S_res, acc.lref)
        for k in cum:
            cum0[k] += tot[k]
            if m >= 1:
                cum[k] += tot[k]
            rec[k + "_cum"] = cum[k]
            rec[k + "_cum_incl0"] = cum0[k]
        if keep_filters and m in (1, M):
            rec["filter"] = dict(lam_ref=acc.lam.cpu().tolist(), S_jump=acc.S_jump.cpu().tolist(),
                                 S_res=acc.S_res.cpu().tolist())
        if diverged:
            status = "diverged"
            records.append(rec)
            break
        eig = kernel_eig(model, X)         # kernel after task m = K_ref for task m+1
        rec.update(boundary_measure(model, X, Yp, eig))
        if m == 0:
            A0, A0_hi = rec["A_raw"], rec["A_hi_raw"]
        rec["A"] = rec["A_raw"] - A0
        rec["A_hi"] = rec["A_hi_raw"] - A0_hi
        records.append(rec)
        del acc
    recs1 = [r for r in records if r["m"] >= 1]
    alphas = [r["alpha"] for r in recs1 if r["alpha"] is not None]
    done = status == "done" and records and records[-1]["m"] == M
    summary = dict(
        A_final=records[-1]["A"] if done else None,
        A_hi_final=records[-1]["A_hi"] if done else None,
        D_cum=records[-1]["D_K_cum"] if done else None,
        D_cum_hi=records[-1]["D_K_hi_cum"] if done else None,
        mean_task_DK=float(np.mean([r["D_K"] for r in recs1])) if recs1 else None,
        alpha_median=float(np.median(alphas)) if alphas else None,
        steps_total=int(sum(r["steps"] for r in records)),
        unfit_total=int(sum(r["unfit"] for r in records)),
        zero_step_micro_total=int(sum(r["zero_step_micro"] for r in records if r["m"] >= 1)),
    )
    del model, opt, eig
    free_memory()
    return _clean(dict(**c, lr=lr, status=status, data_source=DATA_SOURCE, device=str(DEVICE),
                       wall_s=time.time() - t_start, summary=summary, tasks=records))

# --- results file (resumable) ---
def config_signature():
    return dict(n=N_TRAIN, width=WIDTH, lin_width=LIN_WIDTH, C=N_CLASSES, tau=TAU, micro_tau_rule=MICRO_TAU_RULE, s_cap=S_CAP, gn_cap=GN_CAP,
                rho=RHO, rho_hi=RHO_HI, rho_gn=RHO_GN, halvings=GN_MAX_HALVINGS, probe_steps=PROBE_STEPS,
                lr_per_model=LR_PER_MODEL, data_source=DATA_SOURCE)

def load_results():
    sig = config_signature()
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            res = json.load(f)
        if res.get("meta", {}).get("signature") == sig:
            return res
        bak = RESULTS_FILE.replace(".json", f"_old_{int(time.time())}.json")
        os.replace(RESULTS_FILE, bak)
        print(f"!!! {RESULTS_FILE} was produced with a different configuration; moved to {bak}; starting fresh.")
    return dict(meta=dict(signature=sig, mode=MODE, torch=torch.__version__, device=str(DEVICE)),
                lr_selection=None, conditions={})

def save_results(res):
    res["meta"]["mode_last_run"] = MODE
    tmp = RESULTS_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(_clean(res), f)
    os.replace(tmp, RESULTS_FILE)
    mirror(RESULTS_FILE)

# --- learning-rate rule (spec sec. 3); called only in the notebook kernel, never by workers ---
def lr_trial(kind, opt_name, lr, s_cap):
    X, labels, Y0 = get_data(0)
    Y1 = make_targets("R", None, 0, 1, Y0)[1].reshape(-1)
    model = make_model(kind, 0)
    opt = make_optimizer(opt_name, model, lr)
    L0 = Lmax = None
    steps, fit = 0, False
    while True:
        F = outputs(model, X)
        loss = loss_fn(F, Y1)
        Lv = loss.item()
        if L0 is None:
            L0 = Lmax = Lv
        Lmax = max(Lmax, Lv) if math.isfinite(Lv) else float("inf")
        if not math.isfinite(Lv):
            break
        if 2 * Lv <= TAU:                  # mse = 2 * loss
            fit = True
            break
        if steps >= s_cap:
            break
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        steps += 1
    del model, opt
    ok = fit and Lmax <= 10 * L0
    return dict(lr=lr, fit=fit, steps=steps, L0=L0, Lmax=Lmax, ok=ok)

def needed_lr_models():
    kinds = {c["model"] for c in all_conditions()}
    return ["relu"] + sorted(k for k in kinds if k != "relu") if LR_PER_MODEL else ["relu"]

def select_lrs(kinds):
    sel = {}
    for kind in kinds:
        sel[kind] = {}
        for opt_name, grid in (("GD", LR_GRID_GD), ("MOM", LR_GRID_GD), ("ADAM", LR_GRID_ADAM)):
            if kind != "relu" and opt_name == "MOM":
                continue                      # MOM is only used on the ReLU MLP
            trials = [_clean(lr_trial(kind, opt_name, lr, S_CAP)) for lr in grid]
            okv = [t["lr"] for t in trials if t["ok"]]
            chosen = max(okv) if okv else min(grid)
            sel[kind][opt_name] = dict(chosen=chosen, fallback=not okv, trials=trials)
            print(f"  {kind:6s} {opt_name:4s}: " + "  ".join(
                f"{t['lr']:g}:{'ok' if t['ok'] else ('fit' if t['fit'] else 'unfit')}({t['steps']})" for t in trials)
                + f"  ->  chosen {chosen:g}" + ("   !!! NO VALUE QUALIFIED: smallest value used" if not okv else ""))
    return sel

def lr_for(model_kind, learner, lr_sel):
    if learner == "GN":
        return None
    src = lr_sel[model_kind] if (LR_PER_MODEL and model_kind in lr_sel) else lr_sel["relu"]
    base = {"GD": "GD", "GD-half": "GD", "GD-quarter": "GD", "MOM": "MOM", "ADAM": "ADAM", "ADAM-R": "ADAM"}[learner]
    v = src[base]["chosen"]
    return v / 2 if learner == "GD-half" else v / 4 if learner == "GD-quarter" else v

def cond_cost(c, t):
    """Rough cost (seconds) of a condition, for ordering the queue and for time estimates."""
    M, N = c["M"], c["N"]
    if c["learner"] == "GN":
        work = (M + 1) * (3 + N) * t["T_GN"]
    else:
        work = (M + 1) * min(150 * N, N * S_CAP) * t["T_STEP"]
    return (M + 2) * t["T_BOUNDARY"] + work

# --- one result file per finished condition (written by the kernel or by workers; interchangeable) ---
def safe_name(key):
    return re.sub(r"[^A-Za-z0-9.=-]+", "_", key)

def write_json_atomic(path, obj):
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w") as f:
        json.dump(_clean(obj), f)
    os.replace(tmp, path)

def run_subdir(run_dir, sub):
    d = os.path.join(run_dir, sub)
    os.makedirs(d, exist_ok=True)
    return d

def write_condition_file(run_dir, key, out, signature):
    write_json_atomic(os.path.join(run_subdir(run_dir, "done"), safe_name(key) + ".json"),
                      dict(key=key, signature=signature, result=out))

def merge_condition_files(res, run_dir):
    """Copy every per-condition result file with the current configuration into res['conditions']."""
    sig, n_new, n_other = config_signature(), 0, 0
    for p in sorted(glob.glob(os.path.join(run_dir, "done", "*.json"))):
        try:
            with open(p) as f:
                d = json.load(f)
        except Exception:
            continue
        if d.get("signature") != sig:
            n_other += 1
            continue
        if d["key"] not in res["conditions"]:
            n_new += 1
        res["conditions"][d["key"]] = d["result"]
    return n_new, n_other

def list_failed(run_dir):
    """{key: error text} for conditions with a failure record and no result file."""
    out = {}
    for p in sorted(glob.glob(os.path.join(run_dir, "failed", "*.json"))):
        name = os.path.basename(p)[:-5]
        if os.path.exists(os.path.join(run_dir, "done", name + ".json")):
            continue
        try:
            with open(p) as f:
                d = json.load(f)
            out[d["key"]] = d.get("error", "?")
        except Exception:
            pass
    return out

## 10. Parallel workers (library)
Each condition is an independent job. On a machine with several GPUs the notebook starts worker processes (several
per GPU, each pinned to one GPU with `CUDA_VISIBLE_DEVICES` and limited to a few CPU threads). Workers take the next
unclaimed condition from a queue ordered longest-first; a condition is claimed by exclusively creating a claim file
(pid, GPU, start time). Each finished condition is written atomically to its own file in `part5_run/done/`; the kernel
merges these into `results_part5.json`. A claim whose process has died is released and the condition is retried;
a failed condition is recorded in `part5_run/failed/` with its error text and does not stop the others.

In [ ]:
%%writefile -a part5lib.py

# ================= 10. parallel workers =================
def usable_cores():
    try:
        return len(os.sched_getaffinity(0))
    except Exception:
        return os.cpu_count() or 1

def visible_gpu_ids(n):
    """Physical GPU ids for logical GPUs 0..n-1 (respects an existing CUDA_VISIBLE_DEVICES)."""
    env = os.environ.get("CUDA_VISIBLE_DEVICES", "")
    ids = [x.strip() for x in env.split(",") if x.strip()] if env.strip() else [str(i) for i in range(n)]
    return ids[:n]

def est_worker_mem_gb():
    """Rough peak GPU memory of one worker: Jacobian + float64 Gram chunk + vmap buffers + kernel matrices + context."""
    nC = N_TRAIN * N_CLASSES
    P = max(196 * WIDTH + WIDTH + WIDTH * WIDTH + WIDTH + WIDTH * N_CLASSES + N_CLASSES, LIN_WIDTH * N_CLASSES + N_CLASSES)
    b = (4 * nC * P + 8 * nC * min(P, GRAM_CHUNK) + 3 * 4 * min(JAC_CHUNK, nC) * P + 8 * 8 * nC * nC
         + 16 * 4 * P + 4 * 196 * LIN_WIDTH)
    return b / 1e9 + 0.6            # + CUDA context

def resolve_parallel(parallel, num_gpus, workers_per_gpu, threads, cpu_workers):
    cores = usable_cores()
    avail = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
    ng = avail if num_gpus == "auto" else max(0, min(int(num_gpus), avail))
    mem_gb = est_worker_mem_gb()
    res = dict(mode="sequential", slots=[], num_gpus=ng, workers_per_gpu=0, threads=threads, cores=cores,
               mem_per_worker_gb=mem_gb)
    if not parallel:
        res["why"] = "PARALLEL = False"
    elif ng >= 2:
        wpg = min(3, max(1, cores // ng)) if workers_per_gpu == "auto" else max(1, int(workers_per_gpu))
        gpu_mem = min(torch.cuda.get_device_properties(i).total_memory for i in range(ng)) / 1e9
        while wpg > 1 and wpg * mem_gb > 0.7 * gpu_mem:
            wpg -= 1
        ids = visible_gpu_ids(ng)
        res.update(mode="gpu", workers_per_gpu=wpg, gpu_mem_gb=gpu_mem,
                   slots=[ids[g] for k in range(wpg) for g in range(ng)])
        res["why"] = f"{ng} GPUs"
    elif cpu_workers and int(cpu_workers) > 0:
        # on the CPU the BLAS result depends on the thread count, so CPU workers use the kernel's thread count
        # (identical numbers to the sequential path); on a GPU the thread count does not enter the arithmetic
        threads = torch.get_num_threads()
        res.update(mode="cpu", slots=["cpu"] * int(cpu_workers), threads=threads)
        res["why"] = f"CPU_WORKERS = {cpu_workers} (testing mode; {threads} threads each = kernel's, for identical numbers)"
    else:
        res["why"] = f"{ng} GPU(s): single-process path"
    print(f"parallel settings: mode={res['mode']} ({res['why']}) | GPUs used={ng} | workers per GPU={res['workers_per_gpu']}"
          f" | worker processes={len(res['slots'])} | threads per worker={threads} | usable CPU cores={cores}")
    print(f"  estimated GPU memory per worker ~ {mem_gb:.2f} GB"
          + (f" | {res['workers_per_gpu']} per GPU ~ {res['workers_per_gpu'] * mem_gb:.1f} GB of {res['gpu_mem_gb']:.1f} GB"
             f" (limit 70%)" if res["mode"] == "gpu" else ""))
    if res["slots"] and len(res["slots"]) * threads > cores:
        print(f"  note: {len(res['slots'])} workers x {threads} threads > {cores} cores (CPU oversubscription possible)")
    return res

def write_worker_config(run_dir, extra=None):
    """config.json for workers: the CONFIG settings + signature + data source (workers never download or select LRs)."""
    cfg = {k: v for k, v in CFG.items() if k not in ("GRIDS", "TINY_NS")}
    cfg.update(SIGNATURE=config_signature(), FORCE_DATA_SOURCE=DATA_SOURCE, MNIST_DOWNLOAD=False,
               MNIST_ROOT=os.path.abspath(CFG.get("MNIST_ROOT", "./mnist_data")))
    cfg.update(extra or {})
    os.makedirs(run_dir, exist_ok=True)
    write_json_atomic(os.path.join(run_dir, "config.json"), cfg)

def pid_alive(pid):
    try:
        pid = int(pid)
    except Exception:
        return False
    if pid <= 0:
        return False
    if os.name == "nt":
        import ctypes
        k32 = ctypes.windll.kernel32
        h = k32.OpenProcess(0x1000, False, pid)
        if not h:
            return False
        code = ctypes.c_ulong()
        ok = k32.GetExitCodeProcess(h, ctypes.byref(code))
        k32.CloseHandle(h)
        return bool(ok) and code.value == 259
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    except OSError:
        return False
    try:
        with open(f"/proc/{pid}/stat") as f:
            if f.read().rsplit(")", 1)[-1].split()[0] == "Z":
                return False
    except Exception:
        pass
    return True

def _claim_next(run_dir, worker_id, device):
    qdir, cdir, ddir, fdir = (run_subdir(run_dir, s) for s in ("queue", "claims", "done", "failed"))
    for fn in sorted(os.listdir(qdir)):
        if not fn.endswith(".json") or "__" not in fn:
            continue
        name = fn.split("__", 1)[1][:-5]
        if os.path.exists(os.path.join(ddir, name + ".json")) or os.path.exists(os.path.join(fdir, name + ".json")):
            continue
        cpath = os.path.join(cdir, name + ".claim")
        try:
            fd = os.open(cpath, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        except FileExistsError:
            continue
        with os.fdopen(fd, "w") as f:
            json.dump(dict(pid=os.getpid(), worker=worker_id, device=device, host=socket.gethostname(),
                           start=time.time(), session=CFG.get("SESSION")), f)
        try:
            if os.path.exists(os.path.join(ddir, name + ".json")):
                raise FileExistsError
            with open(os.path.join(qdir, fn)) as f:
                job = json.load(f)
        except Exception:
            _remove(cpath)
            continue
        return name, job, cpath
    return None

def _remove(path, tries=100):
    """os.remove that tolerates Windows' 'file in use' (another process may be reading the claim for a moment)."""
    for i in range(tries):
        try:
            os.remove(path)
            return True
        except FileNotFoundError:
            return False
        except PermissionError:
            time.sleep(0.05)
    return False

def worker_main(run_dir, worker_id, device, threads):
    """Entry point of a worker process: pull conditions until none is left unclaimed."""
    with open(os.path.join(run_dir, "config.json")) as f:
        cfg = json.load(f)
    configure(cfg)
    torch.set_num_threads(int(threads))
    set_device("cpu" if device == "cpu" else "cuda")
    print(f"[{worker_id}] pid {os.getpid()} | device {DEVICE} (CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r})"
          f" | torch threads {torch.get_num_threads()} | {time.strftime('%H:%M:%S')}", flush=True)
    fail_key = os.environ.get("PART5_TEST_FAIL_KEY")     # test hook only: makes one named condition fail on purpose
    while True:
        got = _claim_next(run_dir, worker_id, device)
        if got is None:
            break
        name, job, cpath = got
        t0 = time.time()
        print(f"[{worker_id}] start {job['key']} ({time.strftime('%H:%M:%S')})", flush=True)
        try:
            if fail_key and job["key"] == fail_key:
                raise RuntimeError("deliberate failure injected by PART5_TEST_FAIL_KEY (test hook)")
            out = run_condition(job["cond"], job["lr"], M=job.get("M"), s_cap=job.get("s_cap"),
                                gn_cap=job.get("gn_cap"), keep_filters=job.get("keep_filters", True))
            out["worker"] = worker_id
            write_condition_file(run_dir, job["key"], out, cfg["SIGNATURE"])
            print(f"[{worker_id}] done  {job['key']} in {fmt_time(time.time() - t0)}", flush=True)
        except Exception:
            tb = traceback.format_exc()
            print(f"[{worker_id}] FAILED {job['key']}\n{tb}", flush=True)
            write_json_atomic(os.path.join(run_subdir(run_dir, "failed"), name + ".json"),
                              dict(key=job["key"], worker=worker_id, device=device, error=tb, time=time.time()))
        finally:
            _remove(cpath)
        free_memory()
    print(f"[{worker_id}] no unclaimed conditions left; exiting ({time.strftime('%H:%M:%S')})", flush=True)

def _spawn_worker(run_dir, wid, dev, threads):
    env = dict(os.environ)
    env["CUDA_VISIBLE_DEVICES"] = "" if dev == "cpu" else str(dev)
    for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[v] = str(threads)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    libdir = os.path.dirname(os.path.abspath(__file__))
    cmd = [sys.executable, "-u", "-c",
           f"import sys; sys.path.insert(0, {libdir!r}); import part5lib; part5lib.worker_main({run_dir!r}, {wid!r}, {dev!r}, {int(threads)})"]
    logp = os.path.join(run_subdir(run_dir, "logs"), wid + ".log")
    log = open(logp, "a")
    p = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd())
    return dict(p=p, dev=dev, log=log, logp=logp, start=time.time(), end=None)

def _tail(path, n=15):
    try:
        with open(path, errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except Exception:
        return ""

def run_pool(jobs, run_dir, slots, threads, progress_every, timings=None, verbose=True):
    """Run jobs (dicts with key, cond, lr, M, s_cap, gn_cap, keep_filters, cost) in worker processes.
    Interrupting (KeyboardInterrupt) terminates the workers; finished conditions stay in run_dir/done."""
    qdir, cdir, ddir, fdir = (run_subdir(run_dir, s) for s in ("queue", "claims", "done", "failed"))
    for f in glob.glob(os.path.join(qdir, "*")):
        os.remove(f)
    names = {safe_name(j["key"]): j for j in jobs}
    assert len(names) == len(jobs), "file-name collision between condition keys"
    for n in names:
        fp = os.path.join(fdir, n + ".json")
        if os.path.exists(fp):
            os.remove(fp)                                 # failed earlier: retry now
    for rank, j in enumerate(jobs):
        write_json_atomic(os.path.join(qdir, f"{rank:05d}__{safe_name(j['key'])}.json"), j)
    for cf in glob.glob(os.path.join(cdir, "*.claim")):  # claims left by dead processes of an earlier session
        try:
            with open(cf) as f:
                info = json.load(f)
            if not pid_alive(info.get("pid")):
                _remove(cf)
            elif verbose:
                print(f"  note: {os.path.basename(cf)} is held by live pid {info.get('pid')} from session "
                      f"{info.get('session')}; it will be waited for (kill that process to release it)")
        except Exception:
            pass
    procs, counter, respawns, max_respawns = {}, [0], 0, 2 * len(slots) + 5
    slot_of = {}
    def spawn(slot):
        counter[0] += 1
        dev = slots[slot]
        wid = f"w{counter[0]:02d}-{'cpu' if dev == 'cpu' else 'gpu' + str(dev)}"
        procs[wid] = _spawn_worker(run_dir, wid, dev, threads)
        slot_of[wid] = slot
    t0, last = time.time(), 0.0
    for s in range(len(slots)):
        spawn(s)
    poll = max(0.5, min(2.0, progress_every / 2))
    def state():
        st = {}
        for n in names:
            if os.path.exists(os.path.join(ddir, n + ".json")):
                st[n] = "done"
            elif os.path.exists(os.path.join(fdir, n + ".json")):
                st[n] = "failed"
            elif os.path.exists(os.path.join(cdir, n + ".claim")):
                st[n] = "running"
            else:
                st[n] = "pending"
        return st
    def calib():
        r = []
        for n, j in names.items():
            p = os.path.join(ddir, n + ".json")
            if os.path.exists(p) and j.get("cost"):
                try:
                    with open(p) as f:
                        w = json.load(f)["result"].get("wall_s")
                    if w:
                        r.append(w / j["cost"])
                except Exception:
                    pass
        return float(np.median(r)) if r else 1.0
    try:
        while True:
            time.sleep(poll)
            for P in procs.values():
                if P["end"] is None and P["p"].poll() is not None:
                    P["end"] = time.time()
            own = {P["p"].pid: P for P in procs.values()}
            for cf in glob.glob(os.path.join(cdir, "*.claim")):
                try:
                    with open(cf) as f:
                        info = json.load(f)
                except Exception:
                    continue
                pid = info.get("pid")
                dead = (own[pid]["p"].poll() is not None) if pid in own else not pid_alive(pid)
                if dead:
                    if _remove(cf):
                        print(f"  ! worker pid {pid} ({info.get('worker')}) died while running "
                              f"{os.path.basename(cf)[:-6]}; claim released, the condition will be retried", flush=True)
            st = state()
            cnt = {k: sum(v == k for v in st.values()) for k in ("done", "failed", "running", "pending")}
            alive = [w for w, P in procs.items() if P["p"].poll() is None]
            if cnt["pending"] + cnt["running"] == 0:
                break
            if cnt["pending"] > 0:
                busy_slots = {slot_of[w] for w in alive}
                for s in range(len(slots)):
                    if s not in busy_slots and respawns < max_respawns:
                        respawns += 1
                        spawn(s)
                alive = [w for w, P in procs.items() if P["p"].poll() is None]
            if not alive and cnt["running"] == 0:
                print("  !!! no live workers left and conditions remain; stopping the pool (re-run to retry)")
                break
            if verbose and time.time() - last >= progress_every:
                last = time.time()
                per_dev = {}
                for w in alive:
                    d = procs[w]["dev"]
                    per_dev[d] = per_dev.get(d, 0) + 1
                el = time.time() - t0
                left = sum(names[n].get("cost", 0) for n, v in st.items() if v in ("pending", "running"))
                eta = calib() * left / max(1, len(alive)) if timings else None
                print(f"[{time.strftime('%H:%M:%S')}] done {cnt['done']}/{len(names)} | running {cnt['running']} | "
                      f"remaining {cnt['pending']} | failed {cnt['failed']} | workers "
                      + " ".join(f"{'cpu' if d == 'cpu' else 'GPU' + str(d)}:{k}" for d, k in sorted(per_dev.items()))
                      + f" | elapsed {fmt_time(el)}" + (f" | ~{fmt_time(eta)} left (rough)" if eta is not None else ""),
                      flush=True)
        for P in procs.values():
            try:
                P["p"].wait(timeout=120)
            except Exception:
                P["p"].kill()
            if P["end"] is None:
                P["end"] = time.time()
    except BaseException:
        print("\n!!! interrupted: terminating workers ...", flush=True)
        for P in procs.values():
            if P["p"].poll() is None:
                P["p"].terminate()
        for P in procs.values():
            try:
                P["p"].wait(timeout=20)
            except Exception:
                P["p"].kill()
        own = {P["p"].pid for P in procs.values()}
        for cf in glob.glob(os.path.join(cdir, "*.claim")):
            try:
                with open(cf) as f:
                    held = json.load(f).get("pid") in own
                if held:
                    _remove(cf)
            except Exception:
                pass
        st = state()
        print(f"!!! workers stopped. {sum(v == 'done' for v in st.values())} of {len(names)} conditions finished and are "
              f"saved in {ddir}. Re-run the cell to resume.", flush=True)
        raise
    finally:
        for P in procs.values():
            try:
                P["log"].close()
            except Exception:
                pass
    st = state()
    failed = {}
    for n, v in st.items():
        if v == "failed":
            try:
                with open(os.path.join(fdir, n + ".json")) as f:
                    d = json.load(f)
                failed[d["key"]] = dict(error=d.get("error", ""), worker=d.get("worker"),
                                        log=procs.get(d.get("worker"), {}).get("logp"))
            except Exception:
                failed[names[n]["key"]] = dict(error="?", worker=None, log=None)
    per_worker = {w: dict(dev=P["dev"], n=0, cond_wall=0.0, life=(P["end"] or time.time()) - P["start"],
                          exit=P["p"].poll()) for w, P in procs.items()}
    for n, v in st.items():
        if v == "done":
            try:
                with open(os.path.join(ddir, n + ".json")) as f:
                    r = json.load(f)["result"]
                if r.get("worker") in per_worker:
                    per_worker[r["worker"]]["n"] += 1
                    per_worker[r["worker"]]["cond_wall"] += r.get("wall_s") or 0.0
            except Exception:
                pass
    if verbose:
        print(f"\npool finished in {fmt_time(time.time() - t0)}: {sum(v == 'done' for v in st.values())} done, "
              f"{len(failed)} failed, {sum(v in ('pending', 'running') for v in st.values())} not run "
              f"({respawns} worker restarts)")
        for w, d in per_worker.items():
            print(f"  {w:14s} conditions completed {d['n']:3d} | time in conditions {fmt_time(d['cond_wall']):>8s} | "
                  f"process lifetime {fmt_time(d['life']):>8s} | exit code {d['exit']}")
    return dict(state=st, failed=failed, per_worker=per_worker, wall=time.time() - t0)

## 11. Load the library, check the environment, resolve parallel settings
Imports `part5lib.py` (written above), installs the CONFIG settings, then runs the **environment check**: it only
inspects and prints (never installs). It prints versions and the GPU and runs a tiny real GPU smoke test
(float32/float64 tensors, a matmul, a 64×64 `eigh`, one `torch.func` vmap/jacrev call). If the GPU is not usable with
this PyTorch build, it prints the pip command to run yourself (then restart the kernel) and falls back to the CPU.
Finally it decides between the sequential path and parallel workers and prints the resolved settings.

In [ ]:
import os, sys, importlib, socket, time
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import part5lib
part5lib = importlib.reload(part5lib)          # picks up the file just written
CFG["SESSION"] = f"{socket.gethostname()}-{os.getpid()}-{int(time.time())}"
part5lib.configure(CFG)
from part5lib import *                          # library functions and modules (torch, np, ...) into the notebook

part5lib.env_check()
DEVICE = part5lib.DEVICE                         # the library's DEVICE is the single source of truth

if DRIVE_DIR and IN_COLAB and str(DRIVE_DIR).startswith("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as _e:
        print("Drive mount failed:", _e)
if DRIVE_DIR:
    os.makedirs(DRIVE_DIR, exist_ok=True)

for _k in ["relu", "linear"]:
    print(_k, "trainable parameters:", sum(p.numel() for p in trainable(make_model(_k, 0))))
PAR = resolve_parallel(PARALLEL, NUM_GPUS, WORKERS_PER_GPU, THREADS_PER_WORKER, CPU_WORKERS)

## 12. Built-in TESTS (must pass before any grid runs)
1 Jacobian vs finite differences · 2 one GN step cuts ‖r‖ by ≥ 90% · 3 dose bookkeeping ·
4 filter exponents on the linear control (GD ≈ 2, GN ≈ 0) · 5 linear control damage stays 0 ·
6 last micro-target equals Y_m exactly, and (6b) an intermediate micro-target that starts between τ/16 and τ at
N = 4 still takes optimizer steps · 7 reproducibility · 8 the worker-process path gives results identical to the
in-kernel path on the same device. Also downloads the data once (before any worker starts) and measures timings.

Test 2's 90% threshold was a spec error: a full Gauss–Newton step is not expected to remove 90% of the residual in
a nonlinear network, so the supplementary test 2b (the GN step solves the linearised problem, matches it for small
steps, and is exact on the linear control) is the real check; test 2 is still printed but does not block.

In [ ]:
TEST_RESULTS = []
def _report(name, ok, msg):
    TEST_RESULTS.append((name, bool(ok), msg))
    print(f"[{'PASS' if ok else 'FAIL'}] Test {name}: {msg}")

_t_tests = time.time()
Xt, labels_t, Y0t = get_data(0)
nC_t = N_TRAIN * N_CLASSES

# Test 1: Jacobian vs central finite differences (5 random parameters x 5 random outputs)
_m32 = make_model("relu", 0)
_m64 = copy.deepcopy(_m32).double()
_X64 = Xt.double()
_J32 = jacobian(_m32, Xt)
_J64 = jacobian(_m64, _X64)
_g = gen(12345)
_pidx = torch.randperm(_J32.shape[1], generator=_g)[:5].tolist()
_oidx = torch.randperm(nC_t, generator=_g)[:5].tolist()
_th = flat_params(_m64).clone()
_h = 1e-5
_fd = torch.zeros(5, 5, dtype=torch.float64)
with torch.no_grad():
    for b, k in enumerate(_pidx):
        tp = _th.clone(); tp[k] += _h; set_flat_params(_m64, tp); Fp = outputs(_m64, _X64)
        tm = _th.clone(); tm[k] -= _h; set_flat_params(_m64, tm); Fm = outputs(_m64, _X64)
        _fd[:, b] = ((Fp - Fm) / (2 * _h))[_oidx].cpu()
    set_flat_params(_m64, _th)
_blk32 = _J32[_oidx][:, _pidx].double().cpu()
_blk64 = _J64[_oidx][:, _pidx].cpu()
_den = _fd.norm().item()
_rel32 = (_blk32 - _fd).norm().item() / max(_den, 1e-300)
_rel64 = (_blk64 - _fd).norm().item() / max(_den, 1e-300)
_report("1 (Jacobian)", _den > 0 and _rel32 < 1e-2 and _rel64 < 1e-5,
        f"|FD block|={_den:.3e}, rel err float32 J={_rel32:.2e} (<1e-2), float64 J={_rel64:.2e} (<1e-5)")
del _m64, _J32, _J64

# Test 2: one GN step from the initial weights (target Y_0) reduces ||r|| by >= 90%
_m = make_model("relu", 0)
_acc = DoseAcc(kernel_eig(_m, Xt))
with torch.no_grad():
    _r0 = (outputs(_m, Xt) - Y0t.reshape(-1)).double().norm().item()
_it, _fit, _div, _lsf = train_gn(_m, Xt, Y0t.reshape(-1), _acc, cap=1)
with torch.no_grad():
    _r1 = (outputs(_m, Xt) - Y0t.reshape(-1)).double().norm().item()
_report("2 (GN step)", _it == 1 and _r1 <= 0.1 * _r0,
        f"||r|| {_r0:.4e} -> {_r1:.4e} (reduction {100 * (1 - _r1 / _r0):.2f}%, need >= 90%)")

# Test 2b (SUPPLEMENTARY, not in the spec): checks the GN implementation itself, independent of how nonlinear
# the network is over one full step. (i) the step solves the linearised problem: ||r + J dtheta|| / ||r|| < 1e-3;
# (ii) the real network agrees with the linearisation for a small step: | ||r(t)||/||r|| - (1-t) | < 1e-3 at t=0.01;
# (iii) on the linear control (where GN is exact) one GN step removes >= 99.9% of ||r||.
_m = make_model("relu", 0)
_Yf = Y0t.reshape(-1)
_J = jacobian(_m, Xt); _K = gram64(_J)
with torch.no_grad():
    _r = (outputs(_m, Xt) - _Yf).double()
_v = torch.linalg.solve(_K + RHO_GN * torch.linalg.eigvalsh(_K)[-1] * torch.eye(nC_t, dtype=torch.float64, device=DEVICE), _r)
_dth64 = -jt_vec64(_J, _v)
_lin = ((_r + _J.double() @ _dth64).norm() / _r.norm()).item()
_th0 = flat_params(_m)
_curve = {}
with torch.no_grad():
    for _t in (1.0, 0.5, 0.1, 0.01):
        set_flat_params(_m, _th0 + _t * _dth64.float())
        _curve[_t] = ((outputs(_m, Xt) - _Yf).double().norm() / _r.norm()).item()
_ml = make_model("linear", 0)
with torch.no_grad():
    _rl0 = (outputs(_ml, Xt) - _Yf).double().norm().item()
train_gn(_ml, Xt, _Yf, DoseAcc(kernel_eig(_ml, Xt)), cap=1)
with torch.no_grad():
    _rl1 = (outputs(_ml, Xt) - _Yf).double().norm().item()
_ok2b = _lin < 1e-3 and abs(_curve[0.01] - 0.99) < 1e-3 and _rl1 <= 1e-3 * _rl0
_report("2b (SUPPLEMENTARY: GN implementation)", _ok2b,
        f"linearised ||r+J dth||/||r||={_lin:.2e} (<1e-3); real ||r(t)||/||r|| at t=1,0.5,0.1,0.01: "
        f"{', '.join(f'{v:.4f}' for v in _curve.values())} (t=0.01 must be 0.99+-1e-3); "
        f"linear control one GN step: reduction {100 * (1 - _rl1 / _rl0):.4f}% (>=99.9%)")
del _m, _ml, _J, _K

# Test 3: dose bookkeeping for one GD step
_m = make_model("relu", 0)
_eig = kernel_eig(_m, Xt)
_acc = DoseAcc(_eig)
_Y1 = make_targets("R", None, 0, 1, Y0t)[1].reshape(-1)
_opt = make_optimizer("GD", _m, 0.1)
train_first_order(_m, Xt, _Y1, _opt, _acc, cap=1)
_dF = _acc.last_dF
_lref = _eig["lam"][0]
_direct = (_dF @ torch.linalg.solve(_eig["K"] + RHO * _lref * torch.eye(nC_t, dtype=torch.float64, device=DEVICE), _dF)).item()
_accD = _acc.D_K.item()
_rel = abs(_accD - _direct) / abs(_direct)
_report("3 (dose bookkeeping)", _acc.steps == 1 and _rel < 1e-4,
        f"accumulator D_K={_accD:.8e}, direct={_direct:.8e}, rel diff={_rel:.2e} (<1e-4)")

# Tests 4 and 5: linear control
_ml = make_model("linear", 0)
_eta_lin = 0.5 * nC_t / kernel_eig(_ml, Xt)["lam"][0].item()
del _ml
_test_cap = min(S_CAP, 1000)
_out_gd = run_condition(dict(grid="TEST", model="linear", seq="R", eps=None, learner="GD", N=1, seed=0, M=3),
                        lr=_eta_lin, s_cap=_test_cap, keep_filters=False)
_out_gn = run_condition(dict(grid="TEST", model="linear", seq="R", eps=None, learner="GN", N=1, seed=0, M=1),
                        lr=None, keep_filters=False)
_a_gd = [t["alpha"] for t in _out_gd["tasks"] if t["m"] >= 1]
_a_gn = [t["alpha"] for t in _out_gn["tasks"] if t["m"] >= 1]
_ok4 = all(a is not None and 1.9 <= a <= 2.1 for a in _a_gd) and all(a is not None and -0.1 <= a <= 0.1 for a in _a_gn)
_report("4 (filter exponents, linear control)", _ok4,
        f"GD (eta={_eta_lin:.4g}) alpha per task={['%.4f' % a if a is not None else None for a in _a_gd]} in [1.9,2.1]; "
        f"GN alpha={['%.4f' % a if a is not None else None for a in _a_gn]} in [-0.1,0.1]")
_A_lin = [t["A"] for t in _out_gd["tasks"]]
_report("5 (linear control damage)", len(_A_lin) == 4 and max(abs(a) for a in _A_lin) < 1e-4,
        f"after 3 GD tasks: A per task={['%.2e' % a for a in _A_lin]}, max |A|={max(abs(a) for a in _A_lin):.2e} (<1e-4)")

# Test 6: granularity, N=4: last micro-target equals Y_m exactly (sequence E, eps=0.1, task 1)
_YE = make_targets("E", 0.1, 0, 2, Y0t)
_mt = micro_targets(_YE[0], _YE[1], 4)
_ok6 = torch.equal(_mt[-1], _YE[1]) and len(_mt) == 4 and \
    all(torch.allclose(_mt[j - 1], _YE[0] + (j / 4) * (_YE[1] - _YE[0])) for j in range(1, 4))
_report("6 (granularity)", _ok6, f"N=4: last micro-target == Y_m exactly: {torch.equal(_mt[-1], _YE[1])}; "
        f"intermediate targets follow Y_(m-1)+(j/N)(Y_m-Y_(m-1)): {_ok6}")

# Test 6b (spec amendment): with N=4, an intermediate micro-target that starts above tau/16 but below tau takes >= 1
# step. Y_(m-1) = current outputs (residual 0) and Y_m = Y_(m-1) + delta with ||delta||^2/(nC) = 4*TAU, so
# micro-target 1 starts at mse = TAU/4. Run through train_task (the runner's own code path) for GD, ADAM and GN.
for _lrn in ("GD", "ADAM", "GN"):
    _m = make_model("relu", 0)
    with torch.no_grad():
        _Fy = outputs(_m, Xt).reshape(N_TRAIN, N_CLASSES).clone()
    _dl = torch.randn(N_TRAIN, N_CLASSES, generator=gen(777)).to(DEVICE)
    _dl = _dl * math.sqrt(4 * TAU * nC_t) / _dl.norm()
    _mt4 = micro_targets(_Fy, _Fy + _dl, 4)
    with torch.no_grad():
        _mse0 = mse_of(outputs(_m, Xt) - _mt4[0].reshape(-1))
    _opt = None if _lrn == "GN" else make_optimizer(_lrn, _m, 0.1 if _lrn == "GD" else 1e-3)
    _tr = train_task(_m, Xt, _mt4, _lrn, _opt, DoseAcc(kernel_eig(_m, Xt)), S_CAP, GN_CAP)
    _ok6b = (TAU / 16 < _mse0 < TAU) and _tr["micro_steps"][0] >= 1 and _tr["tau_micro"] == TAU / 16 \
        and _tr["tau_last"] == TAU
    _report(f"6b (micro-target threshold, {_lrn})", _ok6b,
            f"N=4: micro-target 1 starts at mse={_mse0:.3e} (tau/16={TAU / 16:.3e} < mse < tau={TAU:.0e}); "
            f"steps per micro-target={_tr['micro_steps']} (first must be >=1); thresholds j<N={_tr['tau_micro']:.3e}, "
            f"j=N={_tr['tau_last']:.0e}; unfit={_tr['unfit']}")
    del _m, _opt, _tr

# Test 7: reproducibility
_c7 = dict(grid="TEST", model="relu", seq="R", eps=None, learner="ADAM", N=1, seed=0, M=2)
_r7a = run_condition(_c7, lr=1e-3, s_cap=min(S_CAP, 200), keep_filters=False)
_r7b = run_condition(_c7, lr=1e-3, s_cap=min(S_CAP, 200), keep_filters=False)
_Aa = [t["A"] for t in _r7a["tasks"]]
_Ab = [t["A"] for t in _r7b["tasks"]]
_d7 = max(abs(a - b) for a, b in zip(_Aa, _Ab)) if len(_Aa) == len(_Ab) else float("inf")
_report("7 (reproducibility)", len(_Aa) == len(_Ab) == 3 and _d7 < 1e-6,
        f"same condition twice: A={['%.6f' % a for a in _Aa]}, max |diff|={_d7:.2e} (<1e-6)")

# Test 8: determinism of the worker path. The test-7 condition is run once in a separate worker process (same device,
# THREADS_PER_WORKER threads) through exactly the machinery the parallel grid uses, and compared with the in-kernel run.
_rd8 = "part5_run_selftest"
shutil.rmtree(_rd8, ignore_errors=True)
write_worker_config(_rd8)
_dev8 = "cpu" if DEVICE.type == "cpu" else visible_gpu_ids(1)[0]
_job8 = dict(key=cond_key(_c7), cond=_c7, lr=1e-3, M=2, s_cap=min(S_CAP, 200), gn_cap=GN_CAP, keep_filters=False, cost=1.0)
_t8 = time.time()
_thr8 = torch.get_num_threads() if _dev8 == "cpu" else THREADS_PER_WORKER   # CPU BLAS rounding depends on thread count
_pool8 = run_pool([_job8], _rd8, [_dev8], _thr8, progress_every=1e9, verbose=False)
_f8 = os.path.join(_rd8, "done", safe_name(_job8["key"]) + ".json")
if os.path.exists(_f8):
    with open(_f8) as _f:
        _A8 = [t["A"] for t in json.load(_f)["result"]["tasks"]]
    _d8 = max(abs(a - b) for a, b in zip(_A8, _Aa)) if len(_A8) == len(_Aa) else float("inf")
    _msg8 = (f"worker process on {'cpu' if _dev8 == 'cpu' else 'GPU ' + _dev8} ({fmt_time(time.time() - _t8)}): "
             f"{_thr8} threads: A={['%.6f' % a for a in _A8]}, in-kernel A={['%.6f' % a for a in _Aa]}, "
             f"max |diff|={_d8:.2e} (<1e-6; bit-identical: {_A8 == _Aa})")
else:
    _d8 = float("inf")
    _msg8 = "worker produced no result: " + "; ".join(v["error"][-300:] for v in _pool8["failed"].values()) + \
            part5lib._tail(glob.glob(os.path.join(_rd8, "logs", "*.log"))[0] if glob.glob(os.path.join(_rd8, "logs", "*.log")) else "")
_report("8 (worker path = in-kernel path)", _d8 < 1e-6, _msg8)

# --- timing calibration (used only for printed time estimates) ---
_m = make_model("relu", 0)
_acc = DoseAcc(kernel_eig(_m, Xt))
_opt = make_optimizer("GD", _m, 1e-6)
_t0 = time.time(); _s, _, _ = train_first_order(_m, Xt, _Y1, _opt, _acc, cap=50)
if DEVICE.type == "cuda": torch.cuda.synchronize()
T_STEP = (time.time() - _t0) / max(_s, 1)
_t0 = time.time(); _e = kernel_eig(_m, Xt); _ = boundary_measure(_m, Xt, probe_targets(0), _e)
T_BOUNDARY = time.time() - _t0
_t0 = time.time(); train_gn(_m, Xt, _Y1, _acc, cap=1)
T_GN = time.time() - _t0
del _m, _acc, _opt, _e
free_memory()
print(f"\ntimings: first-order step {1e3 * T_STEP:.2f} ms | GN iteration {T_GN:.2f} s | "
      f"task-boundary measurement {T_BOUNDARY:.2f} s | tests took {fmt_time(time.time() - _t_tests)}")
_failed = [n for n, ok, _ in TEST_RESULTS if not ok]
_blocking = [n for n in _failed if TEST2_BLOCKING or not n.startswith("2 ")]
if _failed and not _blocking:
    print("\n" + "!" * 100 + "\n!!! Test 2 (literal spec threshold: one full GN step removes >= 90% of ||r||) FAILED, but Test 2b shows\n"
          "!!! the GN step is implemented correctly: the shortfall is the network's nonlinearity over one full step.\n"
          "!!! GN in the grids uses the line search and up to GN_CAP iterations, so it still fits each target.\n"
          "!!! Continuing because TEST2_BLOCKING = False (set it to True in CONFIG to stop here instead).\n" + "!" * 100)
assert not _blocking, f"TESTS FAILED: {_blocking}. Do not run the grids until this is fixed."
print(f"{len(TEST_RESULTS) - len(_failed)} of {len(TEST_RESULTS)} TESTS PASSED" +
      ("" if not _failed else f"; non-blocking failure: {_failed}"))
DATA_SOURCE = part5lib.DATA_SOURCE              # decided once, here in the kernel (workers reuse it, never download)
for _s in SEEDS:
    get_data(_s)                                # all seeds' data loaded / downloaded before any worker starts

## 13. Learning-rate selection (done once, before any plasticity measurement, then frozen)
Rule (spec §3): seed 0, sequence R, N = 1; from the initial weights train ONLY task 1 with each grid value;
choose the largest value that reaches Fit within S_cap and whose loss never exceeds 10× its starting value.
ADAM-R uses ADAM's value; GD-half / GD-quarter use η_GD/2 and η_GD/4.

*Note (a decision, not in the spec):* the spec does not say which learning rate the tanh arm and the linear
control use. With `LR_PER_MODEL = True` the identical rule is applied to each of those models (seed 0, task 1),
because a value chosen on the ReLU MLP can be unstable on the 2048-feature linear control.
The choice is recorded in the results file and is reused if the notebook is re-run.
Learning rates are selected only here, in the notebook kernel; workers receive them with each condition.

In [ ]:
RESULTS = load_results()
RESULTS["meta"]["tests_last_run"] = [dict(test=n, passed=ok, detail=msg) for n, ok, msg in TEST_RESULTS]
save_results(RESULTS)
LR_SEL = RESULTS.get("lr_selection") or {}
_missing = [k for k in needed_lr_models() if k not in LR_SEL]
if LR_SEL:
    print(f"Learning rates already selected earlier for {sorted(LR_SEL)} (reused, frozen).")
if _missing:
    print(f"Selecting learning rates for {_missing} (value:outcome(steps)):")
    _t0 = time.time()
    LR_SEL.update(select_lrs(_missing))
    RESULTS["lr_selection"] = LR_SEL
    save_results(RESULTS)
    print(f"LR selection took {fmt_time(time.time() - _t0)}")

for _k, _d in LR_SEL.items():
    print(f"CHOSEN [{_k}]: " + ", ".join(f"{o}={d['chosen']:g}" for o, d in _d.items()))

## 14. Run the grids (resumable; sequential or parallel)
Finished conditions (in `results_part5.json` or as files in `part5_run/done/`) are skipped. With one GPU (or none,
or `PARALLEL = False`) conditions run one after another in this kernel, as before. With several GPUs they run in
worker processes; a progress line is printed every `PROGRESS_EVERY_S` seconds. **Interrupt this cell to stop**: the
workers are terminated and finished conditions stay on disk; run the cell again to resume.
Time estimates are rough (timings measured in the TESTS cell, calibrated on finished conditions).

In [ ]:
_TIMINGS = dict(T_STEP=T_STEP, T_GN=T_GN, T_BOUNDARY=T_BOUNDARY)
_CALIB = []   # (observed / prior) for finished conditions (sequential path)

def estimate(c):
    f = float(np.median(_CALIB)) if _CALIB else 1.0
    return f * cond_cost(c, _TIMINGS)

RESULTS = load_results()
RESULTS["lr_selection"] = LR_SEL
_new, _other = merge_condition_files(RESULTS, RUN_DIR)
save_results(RESULTS)
if _new or _other:
    print(f"merged {_new} condition file(s) from {RUN_DIR}/done" +
          (f"; ignored {_other} made with a different configuration" if _other else ""))
_conds = all_conditions()
_todo = [c for c in _conds if cond_key(c) not in RESULTS["conditions"]]
print(f"{len(_conds)} conditions in MODE={MODE}: {len(_conds) - len(_todo)} already done, {len(_todo)} to run "
      f"({PAR['mode']} mode)")
POOL_FAILED = {}
_t_all = time.time()
_f = lambda v, p=".4g": ("NA" if v is None else format(v, p))
if not _todo:
    print("nothing to run.")
elif PAR["mode"] == "sequential":
    for g in GRIDS:
        gc_list = [c for c in _conds if c["grid"] == g["name"]]
        if not gc_list:
            print(f"\n=== {g['name']}: no conditions in MODE={MODE} ===")
            continue
        todo = [c for c in gc_list if cond_key(c) not in RESULTS["conditions"]]
        print(f"\n=== {g['name']}: {len(gc_list)} conditions, {len(gc_list) - len(todo)} already done; "
              f"estimated time for the rest ~ {fmt_time(sum(estimate(c) for c in todo))} (rough) ===")
        for c in todo:
            k = cond_key(c)
            out = run_condition(c, lr_for(c["model"], c["learner"], LR_SEL))
            out["worker"] = "kernel"
            write_condition_file(RUN_DIR, k, out, config_signature())
            RESULTS["conditions"][k] = out
            save_results(RESULTS)
            _CALIB.append(out["wall_s"] / max(cond_cost(c, _TIMINGS), 1e-9))
            s = out["summary"]
            print(f"  {k:60s} {out['status']:8s} {fmt_time(out['wall_s']):>8s} | A_final={_f(s['A_final'])} "
                  f"D_cum={_f(s['D_cum'])} steps={s['steps_total']} unfit={s['unfit_total']} "
                  f"zero-step={s['zero_step_micro_total']}")
            free_memory()
else:
    _jobs = [dict(key=cond_key(c), cond=c, lr=lr_for(c["model"], c["learner"], LR_SEL), M=None, s_cap=None, gn_cap=None,
                  keep_filters=True, cost=cond_cost(c, _TIMINGS)) for c in _todo]
    _jobs.sort(key=lambda j: -j["cost"])                      # longest first
    for g in GRIDS:
        n_g = sum(j["cond"]["grid"] == g["name"] for j in _jobs)
        if n_g:
            print(f"  {g['name']:7s}: {n_g:3d} conditions, ~{fmt_time(sum(j['cost'] for j in _jobs if j['cond']['grid'] == g['name']))}"
                  f" of single-worker time")
    print(f"estimated time with {len(PAR['slots'])} workers ~ {fmt_time(sum(j['cost'] for j in _jobs) / len(PAR['slots']))}"
          f" (rough; the longest single condition ~ {fmt_time(_jobs[0]['cost'])})")
    write_worker_config(RUN_DIR)
    try:
        _pool = run_pool(_jobs, RUN_DIR, PAR["slots"], PAR["threads"], PROGRESS_EVERY_S, timings=_TIMINGS)
        POOL_FAILED = _pool["failed"]
    finally:
        RESULTS = load_results()
        RESULTS["lr_selection"] = LR_SEL
        merge_condition_files(RESULTS, RUN_DIR)
        save_results(RESULTS)
    for k in sorted(RESULTS["conditions"]):
        if k in {j["key"] for j in _jobs}:
            s = RESULTS["conditions"][k]["summary"]
            print(f"  {k:60s} {RESULTS['conditions'][k]['status']:8s} {fmt_time(RESULTS['conditions'][k]['wall_s']):>8s} | "
                  f"A_final={_f(s['A_final'])} D_cum={_f(s['D_cum'])} steps={s['steps_total']} unfit={s['unfit_total']} "
                  f"zero-step={s['zero_step_micro_total']} | {RESULTS['conditions'][k].get('worker')}")
if POOL_FAILED:
    print("\n" + "X" * 100 + f"\nXXX {len(POOL_FAILED)} CONDITION(S) FAILED (the others finished; re-run this cell to retry):")
    for k, v in POOL_FAILED.items():
        print(f"XXX {k}  (worker {v['worker']}, log {v['log']})\n" + "\n".join("XXX   " + l for l in v["error"].strip().splitlines()[-6:]))
    print("X" * 100)
print(f"\nGrid cell finished in {fmt_time(time.time() - _t_all)} (this session).")

## 15. Analysis: pre-registered predictions H1–H9 and report-only items R1–R7
Slopes are OLS in log-log per seed, then the median over seeds. **PASS** needs the rule to hold for the median
and for at least 2 of 3 seeds (for fewer seeds: all of them); **INCONCLUSIVE** if it holds for the median only.
For pooled rules (H6, H8) the pooled statistic plays the role of the median and each seed is also checked alone.
In "quick" mode every verdict is marked PRELIMINARY.

In [ ]:
RES = load_results()
merge_condition_files(RES, RUN_DIR)
CONDS = RES["conditions"]
_fails = {k: v for k, v in list_failed(RUN_DIR).items() if k not in CONDS}
_missing = [cond_key(c) for c in all_conditions() if cond_key(c) not in CONDS]
if _fails or _missing:
    print("X" * 100 + f"\nXXX ANALYSIS RUNS ON WHAT FINISHED: {len(_missing)} planned condition(s) have no result"
          + (f", {len(_fails)} of them FAILED:" if _fails else ":"))
    for k in _missing:
        print("XXX   " + k + ("   <-- FAILED: " + _fails[k].strip().splitlines()[-1] if k in _fails else ""))
    print("X" * 100)
PRELIM = MODE != "full"
VERDICTS = {}

def getc(grid, model, seq, learner, N, seed, eps=None):
    c = CONDS.get(cond_key(dict(grid=grid, model=model, seq=seq, eps=eps, learner=learner, N=N, seed=seed, M=M_TASKS)))
    return c if (c is not None and c.get("status") == "done") else None

def tasks1(c):
    return [t for t in c["tasks"] if t["m"] >= 1]

def S(c, field):
    return None if c is None else c["summary"].get(field)

def ols_loglog(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 2 or np.any(~np.isfinite(y)) or np.any(y <= 0):
        return None
    return float(np.polyfit(np.log(x), np.log(y), 1)[0])

def spear(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3 or np.all(x[m] == x[m][0]) or np.all(y[m] == y[m][0]):
        return None
    if HAVE_SCIPY:
        rho_s = float(stats.spearmanr(x[m], y[m])[0])
    else:
        def _rank(v):   # average ranks for ties (same definition as scipy.stats.rankdata)
            order = np.argsort(v, kind="mergesort"); r = np.empty(len(v)); r[order] = np.arange(1, len(v) + 1)
            for val in np.unique(v):
                idx = v == val
                r[idx] = r[idx].mean()
            return r
        rho_s = float(np.corrcoef(_rank(x[m]), _rank(y[m]))[0, 1])
    return round(rho_s, 10)   # rounding: 0.8 must compare as 0.8, not 0.7999999999

def med(vals):
    v = [x for x in vals if x is not None]
    return float(np.median(v)) if v else None

def verdict(hid, median_ok, seed_oks, note=""):
    if median_ok is None:
        v = "NOT RUN"
    else:
        k, n = sum(bool(s) for s in seed_oks), len(seed_oks)
        need = 2 if n >= 3 else n
        v = "PASS" if (median_ok and k >= need) else ("INCONCLUSIVE" if median_ok else "FAIL")
        if PRELIM:
            v += " (PRELIMINARY)"
    VERDICTS[hid] = v + (f" [{note}]" if note else "")
    print(f"  ==> {hid}: {VERDICTS[hid]}\n")

def f4(v):
    return "NA" if v is None else (f"{v:.4g}" if isinstance(v, (int, float)) else str(v))

def inrange(v, lo, hi):
    return v is not None and lo <= v <= hi

def grid_Ns(name):
    return sorted({c["N"] for c in all_conditions() if c["grid"] == name})

def grid_eps(name):
    return sorted({c["eps"] for c in all_conditions() if c["grid"] == name and c["eps"] is not None})

# ---------- H1 ----------
print("H1  GN, sequence R: per-task dose falls as 1/N.  Rule: slope of log(mean per-task D_K) on log N in [-1.3, -0.7]")
_Ns = grid_Ns("G-gn")
_sl = {}
for s in SEEDS:
    xs, ys = [], []
    for N in _Ns:
        c = getc("G-gn", "relu", "R", "GN", N, s)
        if c is not None:
            xs.append(N); ys.append(S(c, "mean_task_DK"))
    _sl[s] = ols_loglog(xs, ys)
    print(f"  seed {s}: N={xs} mean per-task D_K={[f4(y) for y in ys]}  slope={f4(_sl[s])}")
_m = med(_sl.values())
print(f"  median slope = {f4(_m)}")
verdict("H1", None if _m is None else inrange(_m, -1.3, -0.7), [inrange(v, -1.3, -0.7) for v in _sl.values()])

# ---------- H2 ----------
print("H2  GN, sequence R: damage falls with N.  Rule: A_final strictly decreasing in N (median over seeds) "
      "and A_final(N=16) <= 0.5 A_final(N=1)")
_Nhi = 16
_note = ""
if 16 not in _Ns and MODE == "tiny" and _Ns:
    _Nhi = max(_Ns); _note = f"tiny mode: N={_Nhi} stands in for N=16"
def _h2_ok(Afs):
    if any(Afs.get(N) is None for N in _Ns) or 1 not in Afs or _Nhi not in Afs:
        return None
    seq = [Afs[N] for N in _Ns]
    return all(b < a for a, b in zip(seq, seq[1:])) and Afs[_Nhi] <= 0.5 * Afs[1]
_per = {}
for s in SEEDS:
    _per[s] = {N: S(getc("G-gn", "relu", "R", "GN", N, s), "A_final") for N in _Ns}
    print(f"  seed {s}: A_final by N = {{{', '.join(f'{N}: {f4(v)}' for N, v in _per[s].items())}}}  ok={_h2_ok(_per[s])}")
_medA = {N: med([_per[s][N] for s in SEEDS]) for N in _Ns}
print(f"  median A_final by N = {{{', '.join(f'{N}: {f4(v)}' for N, v in _medA.items())}}}")
verdict("H2", _h2_ok(_medA), [_h2_ok(_per[s]) for s in SEEDS], _note)

# ---------- H3 ----------
print("H3  GN, sequence E(eps): dose ~ eps^2, damage ~ eps^2.  Rule: slope log(mean per-task D_K) vs log eps in [1.6,2.4]; "
      "slope log A_final vs log eps in [1.2,2.8]")
_eps = grid_eps("G-eps")
for lrn in ["GN", "GD", "ADAM"]:
    _sd, _sa = {}, {}
    for s in SEEDS:
        cs = [(e, getc("G-eps", "relu", "E", lrn, 1, s, eps=e)) for e in _eps]
        cs = [(e, c) for e, c in cs if c is not None]
        _sd[s] = ols_loglog([e for e, _ in cs], [S(c, "mean_task_DK") for _, c in cs])
        _sa[s] = ols_loglog([e for e, _ in cs], [S(c, "A_final") for _, c in cs])
        print(f"  {lrn:4s} seed {s}: eps={[e for e, _ in cs]} mean D_K={[f4(S(c, 'mean_task_DK')) for _, c in cs]} "
              f"A_final={[f4(S(c, 'A_final')) for _, c in cs]}  slope_D={f4(_sd[s])} slope_A={f4(_sa[s])}")
    if lrn == "GN":
        mD, mA = med(_sd.values()), med(_sa.values())
        print(f"  GN median slopes: dose {f4(mD)}, damage {f4(mA)}  (slope_A is undefined if any A_final <= 0)")
        verdict("H3", None if (mD is None and mA is None) else (inrange(mD, 1.6, 2.4) and inrange(mA, 1.2, 2.8)),
                [inrange(_sd[s], 1.6, 2.4) and inrange(_sa[s], 1.2, 2.8) for s in SEEDS])
print("  (GD and ADAM slopes above are report-only)\n")

# ---------- H4 ----------
print("H4  filter exponents (median over tasks 1..M of the per-task alpha), N=1. Rule: GD in [1.5,2.5]; GN in [-0.5,0.5]")
_aGD = {s: S(getc("G-main", "relu", "R", "GD", 1, s), "alpha_median") for s in SEEDS}
_aGN = {s: S(getc("G-gn", "relu", "R", "GN", 1, s), "alpha_median") for s in SEEDS}
for s in SEEDS:
    print(f"  seed {s}: alpha GD={f4(_aGD[s])}  GN={f4(_aGN[s])}")
for lrn, grid in [("MOM", "G-main"), ("ADAM", "G-main"), ("ADAM-R", "G-main"), ("GD-half", "G-lr"), ("GD-quarter", "G-lr")]:
    print(f"  (report) {lrn:10s} alpha by seed: {[f4(S(getc(grid, 'relu', 'R', lrn, 1, s), 'alpha_median')) for s in SEEDS]}")
for lrn, grid in [("GD", "G-main"), ("MOM", "G-main"), ("ADAM", "G-main"), ("ADAM-R", "G-main"), ("GN", "G-gn")]:
    print(f"  (report) {lrn:6s} alpha (seed {SEEDS[0]}) by N: " +
          ", ".join(f"N={N}: {f4(S(getc(grid, 'relu', 'R', lrn, N, SEEDS[0]), 'alpha_median'))}" for N in grid_Ns(grid)))
_mg, _mn = med(_aGD.values()), med(_aGN.values())
print(f"  median alpha: GD {f4(_mg)}, GN {f4(_mn)}")
verdict("H4", None if (_mg is None or _mn is None) else (inrange(_mg, 1.5, 2.5) and inrange(_mn, -0.5, 0.5)),
        [inrange(_aGD[s], 1.5, 2.5) and inrange(_aGN[s], -0.5, 0.5) for s in SEEDS])

# ---------- H5 (and R6 part) ----------
FIVE = [("GD", "G-main"), ("MOM", "G-main"), ("ADAM", "G-main"), ("ADAM-R", "G-main"), ("GN", "G-gn")]
def h5(dfield, afield, label):
    print(f"{label}  optimizer ordering at N=1 follows dose. Rule: Spearman({dfield}, {afield}) across the five learners "
          f"(median over seeds) >= 0.8")
    rs = {}
    for s in SEEDS:
        cs = [(l, getc(g, "relu", "R", l, 1, s)) for l, g in FIVE]
        cs = [(l, c) for l, c in cs if c is not None and S(c, afield) is not None]
        rs[s] = spear([S(c, dfield) for _, c in cs], [S(c, afield) for _, c in cs]) if len(cs) >= 3 else None
        print(f"  seed {s}: " + ", ".join(f"{l}: D={f4(S(c, dfield))} A={f4(S(c, afield))}" for l, c in cs)
              + f"  | n_learners={len(cs)} Spearman={f4(rs[s])}")
    m = med(rs.values())
    print(f"  median Spearman = {f4(m)}")
    return m, rs
_m, _rs = h5("D_cum", "A_final", "H5")
verdict("H5", None if _m is None else _m >= 0.8, [r is not None and r >= 0.8 for r in _rs.values()])

# ---------- H6 (and R6 part) ----------
def pooled_records(grids, seeds):
    out = []
    for c in CONDS.values():
        if c.get("status") == "done" and c["grid"] in grids and c["seed"] in seeds and c["M"] == M_TASKS:
            for t in tasks1(c):
                out.append(dict(t, learner=c["learner"], N=c["N"], seed=c["seed"]))
    return out

def h6(dkey, akey, label):
    print(f"{label}  collapse (pooled G-main + G-gn records, tasks 1..M). Rule: Spearman(D_cum, A) >= 0.8 and larger than "
          f"Spearman(A, task index), Spearman(A, cum W_move), Spearman(A, cum D_raw), Spearman(A, ||theta||)")
    def one(recs):
        A = [r[akey] for r in recs]
        vals = dict(dose=spear([r[dkey] for r in recs], A), task=spear(A, [r["m"] for r in recs]),
                    W_move=spear(A, [r["W_move_cum"] for r in recs]), D_raw=spear(A, [r["D_raw_cum"] for r in recs]),
                    theta=spear(A, [r["theta_norm"] for r in recs]))
        others = [vals[k] for k in ("task", "W_move", "D_raw", "theta")]
        ok = None if vals["dose"] is None else (vals["dose"] >= 0.8 and all(o is None or vals["dose"] > o for o in others))
        return vals, ok, len(recs)
    vals, ok, n = one(pooled_records({"G-main", "G-gn"}, SEEDS))
    print(f"  pooled (n={n}): " + ", ".join(f"{k}={f4(v)}" for k, v in vals.items()) + f"  -> {ok}")
    sok = []
    for s in SEEDS:
        v, o, n = one(pooled_records({"G-main", "G-gn"}, [s]))
        sok.append(bool(o))
        print(f"  seed {s} (n={n}): " + ", ".join(f"{k}={f4(x)}" for k, x in v.items()) + f"  -> {o}")
    return ok, sok
_ok, _sok = h6("D_K_cum", "A", "H6")
verdict("H6", _ok, _sok)

# ---------- H7 ----------
print("H7  dose intervention, GD / GD-half / GD-quarter at N=1. Rule: D_cum and A_final both decrease as eta is "
      "lowered (GD > GD-half > GD-quarter), in the median over seeds")
def _h7(d):
    if any(v is None for v in d["D"] + d["A"]):
        return None
    return d["D"][0] > d["D"][1] > d["D"][2] and d["A"][0] > d["A"][1] > d["A"][2]
_per = {}
for s in SEEDS:
    cs = [getc("G-main", "relu", "R", "GD", 1, s), getc("G-lr", "relu", "R", "GD-half", 1, s),
          getc("G-lr", "relu", "R", "GD-quarter", 1, s)]
    _per[s] = dict(D=[S(c, "D_cum") for c in cs], A=[S(c, "A_final") for c in cs])
    print(f"  seed {s}: D_cum (eta, eta/2, eta/4) = {[f4(v) for v in _per[s]['D']]}  A_final = {[f4(v) for v in _per[s]['A']]}"
          f"  ok={_h7(_per[s])}")
_md = dict(D=[med([_per[s]["D"][i] for s in SEEDS]) for i in range(3)],
           A=[med([_per[s]["A"][i] for s in SEEDS]) for i in range(3)])
print(f"  median: D_cum={[f4(v) for v in _md['D']]}  A_final={[f4(v) for v in _md['A']]}")
verdict("H7", _h7(_md), [_h7(_per[s]) for s in SEEDS])

# ---------- H8 ----------
print("H8  damage means lost trainability (pooled G-main + G-gn, tasks 1..M). Rule: Spearman(A, log psi) >= 0.6 and "
      "Spearman(psi, psi_lin) >= 0.8")
def _h8(recs):
    A = [r["A"] for r in recs]
    lp = [math.log(r["psi"]) if (r.get("psi") is not None and r["psi"] > 0) else float("nan") for r in recs]
    psi = [r["psi"] if r.get("psi") is not None else float("nan") for r in recs]
    pl = [r["psi_lin"] if r.get("psi_lin") is not None else float("nan") for r in recs]
    a, b = spear(A, lp), spear(psi, pl)
    return a, b, (None if (a is None or b is None) else (a >= 0.6 and b >= 0.8)), len(recs)
_a, _b, _ok, _n = _h8(pooled_records({"G-main", "G-gn"}, SEEDS))
print(f"  pooled (n={_n}): Spearman(A, log psi)={f4(_a)}  Spearman(psi, psi_lin)={f4(_b)}  -> {_ok}")
_sok = []
for s in SEEDS:
    a, b, o, n = _h8(pooled_records({"G-main", "G-gn"}, [s]))
    _sok.append(bool(o))
    print(f"  seed {s} (n={n}): {f4(a)}, {f4(b)} -> {o}")
verdict("H8", _ok, _sok)

# ---------- H9 ----------
print("H9  linear control (G-lin): |A| < 1e-3 at every task for every learner")
_mx = {}
for s in SEEDS:
    vals = []
    for lrn in ["GD", "ADAM", "GN"]:
        c = getc("G-lin", "linear", "R", lrn, 1, s)
        if c is not None:
            mA = max(abs(t["A"]) for t in c["tasks"] if t.get("A") is not None)
            vals.append(mA)
            print(f"  seed {s} {lrn:5s}: max|A| over tasks = {mA:.3e}  (unfit micro-targets: {S(c, 'unfit_total')})")
    _mx[s] = max(vals) if len(vals) == 3 else None
_m = med(_mx.values())
verdict("H9", None if _m is None else _m < 1e-3, [v is not None and v < 1e-3 for v in _mx.values()])

# ---------- report-only ----------
print("=" * 100 + "\nREPORT-ONLY ITEMS (no verdicts)\n" + "=" * 100)
print("R1  band-dose shares of cumulative D_K (TOP 10% / MID 80% / BOT 10% eigen-directions of K_ref), seed", SEEDS[0])
for lrn, grid in FIVE + [("GD-half", "G-lr"), ("GD-quarter", "G-lr")]:
    for N in grid_Ns(grid):
        c = getc(grid, "relu", "R", lrn, N, SEEDS[0])
        if c is None:
            continue
        tb = {b: sum(t["D_band"][b] for t in tasks1(c)) for b in ("TOP", "MID", "BOT")}
        tot = sum(tb.values())
        print(f"  {lrn:10s} N={N:3d}: " + "  ".join(f"{b}={tb[b] / tot:.3f}" for b in tb) if tot > 0 else f"  {lrn} N={N}: zero dose")

print("\nR2  ADAM vs ADAM-R (seed %d): D_cum, A_final, mean first-step ||dtheta|| and ||dF|| after a target change" % SEEDS[0])
for N in grid_Ns("G-main"):
    for lrn in ["ADAM", "ADAM-R"]:
        c = getc("G-main", "relu", "R", lrn, N, SEEDS[0])
        if c is None:
            continue
        fd = [t["first_step_dtheta"] for t in tasks1(c) if t.get("first_step_dtheta") is not None]
        ff = [t["first_step_dF"] for t in tasks1(c) if t.get("first_step_dF") is not None]
        print(f"  N={N:3d} {lrn:6s}: D_cum={f4(S(c, 'D_cum'))}  A_final={f4(S(c, 'A_final'))}  "
              f"first ||dtheta||={f4(float(np.mean(fd)) if fd else None)}  first ||dF||={f4(float(np.mean(ff)) if ff else None)}")

print("\nR3  unit statistics vs D_cum and vs A (pooled G-main + G-gn, Spearman)")
_pr = pooled_records({"G-main", "G-gn"}, SEEDS)
for name, fn in [("dead frac (last layer)", lambda r: r["dead_frac"][-1]), ("dead frac (layer 1)", lambda r: r["dead_frac"][0]),
                 ("duplicate pairs", lambda r: r["dup_pairs"]), ("stable rank", lambda r: r["stable_rank"]),
                 ("||theta||", lambda r: r["theta_norm"])]:
    v = [fn(r) for r in _pr]
    print(f"  {name:24s}: vs D_cum {f4(spear(v, [r['D_K_cum'] for r in _pr]))}   vs A {f4(spear(v, [r['A'] for r in _pr]))}"
          f"   range [{f4(min(v) if v else None)}, {f4(max(v) if v else None)}]")

print("\nR4  optimizer steps (GN: iterations) needed per task, tasks 0..M (seed %d)" % SEEDS[0])
for lrn, grid in FIVE + [("GD-half", "G-lr"), ("GD-quarter", "G-lr")]:
    for N in grid_Ns(grid):
        c = getc(grid, "relu", "R", lrn, N, SEEDS[0])
        if c is not None:
            print(f"  {lrn:10s} N={N:3d}: {[t['steps'] for t in c['tasks']]}")

print("\nR5  tanh arm vs ReLU (seed %d): A_final / D_cum" % SEEDS[0])
for lrn in ["GD", "ADAM", "GN"]:
    for N in grid_Ns("G-tanh"):
        ct = getc("G-tanh", "tanh", "R", lrn, N, SEEDS[0])
        cr = getc("G-gn" if lrn == "GN" else "G-main", "relu", "R", lrn, N, SEEDS[0])
        print(f"  {lrn:5s} N={N:3d}: tanh A_final={f4(S(ct, 'A_final'))} D_cum={f4(S(ct, 'D_cum'))} | "
              f"relu A_final={f4(S(cr, 'A_final'))} D_cum={f4(S(cr, 'D_cum'))}")
if not grid_Ns("G-tanh"):
    print("  (G-tanh not run in this MODE)")

print("\nR6  sensitivity of H5 and H6 to rho (rho = 1e-2 versions of dose and damage)")
_m, _rs = h5("D_cum_hi", "A_hi_final", "R6/H5(rho=1e-2)")
print(f"  R6/H5 result: median Spearman {f4(_m)} (would pass: {None if _m is None else _m >= 0.8})")
_ok, _sok = h6("D_K_hi_cum", "A_hi", "R6/H6(rho=1e-2)")
print(f"  R6/H6 result: pooled rule holds: {_ok}; seeds: {_sok}")

print("\nR7  anomalies (measurements only)")
_anom = 0
for k, c in CONDS.items():
    if c.get("M") != M_TASKS:
        continue
    msgs = []
    if c.get("status") != "done":
        msgs.append(f"status={c.get('status')}")
    uf = sum(t["unfit"] for t in c["tasks"])
    if uf:
        msgs.append(f"UNFIT micro-targets={uf}")
    ls = sum(t.get("gn_line_search_fail", 0) for t in c["tasks"])
    if ls:
        msgs.append(f"GN line-search failures={ls}")
    zs = sum(t.get("zero_step_micro", 0) for t in tasks1(c))
    if zs:
        msgs.append(f"micro-targets already fit with zero steps={zs} (thresholds: j<N mse<=tau/N^2, j=N mse<=tau)")
    As = [t["A"] for t in c["tasks"] if t.get("A") is not None]
    dec = sum(1 for a, b in zip(As, As[1:]) if b < a)
    if dec:
        msgs.append(f"A decreased at {dec} task(s)")
    bad = [t["m"] for t in c["tasks"] if t.get("psi") is None or (t.get("psi") is not None and t["psi"] > 1)]
    if bad:
        msgs.append(f"probe psi>1 or undefined at tasks {bad}")
    if msgs:
        _anom += 1
        print(f"  {k}: " + "; ".join(msgs))
for kind, d in LR_SEL.items():
    for o, v in d.items():
        if v.get("fallback"):
            print(f"  LR rule: no grid value qualified for {kind}/{o}; smallest value used")
if not _anom:
    print("  none")

print("\n" + "=" * 100 + "\nVERDICTS" + (" (PRELIMINARY: MODE is not 'full')" if PRELIM else "") + "\n" + "=" * 100)
for h, v in VERDICTS.items():
    print(f"  {h}: {v}")

## 16. Figures
(a) damage vs task at N = 1 · (b) final damage vs N · (c) collapse: damage vs cumulative dose ·
(d) filter plot S_jump/S_res vs λ · (e) band-dose shares · (f) probe ψ vs damage · (g) dose and damage vs ε.
Saved as PNG in `figures_part5/` and shown below.

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False
COLORS = {"GD": "#1f77b4", "MOM": "#ff7f0e", "ADAM": "#2ca02c", "ADAM-R": "#d62728", "GN": "#9467bd",
          "GD-half": "#8c564b", "GD-quarter": "#e377c2"}
FIG_FILES = []

def savefig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    FIG_FILES.append(path)
    mirror(path)
    plt.show()
    plt.close(fig)

def fig_a():
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), squeeze=False); ax = ax[0, 0]
    for lrn, grid in FIVE + [("GD-half", "G-lr"), ("GD-quarter", "G-lr")]:
        for s in SEEDS:
            c = getc(grid, "relu", "R", lrn, 1, s)
            if c is not None:
                ax.plot([t["m"] for t in c["tasks"]], [t["A"] for t in c["tasks"]], "-o", ms=3, color=COLORS[lrn],
                        alpha=0.8, label=lrn if s == SEEDS[0] else None)
    ax.set_xlabel("task index m"); ax.set_ylabel("damage A"); ax.set_title("(a) damage vs task, N = 1"); ax.legend(fontsize=7)
    savefig(fig, "fig_a_damage_vs_task.png")

def fig_b():
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), squeeze=False); ax = ax[0, 0]
    for lrn, grid in FIVE:
        Ns = grid_Ns(grid)
        ys = [med([S(getc(grid, "relu", "R", lrn, N, s), "A_final") for s in SEEDS]) for N in Ns]
        pts = [(N, y) for N, y in zip(Ns, ys) if y is not None]
        if pts:
            ax.plot([p[0] for p in pts], [p[1] for p in pts], "-o", color=COLORS[lrn], label=lrn)
    ax.set_xscale("log", base=2); ax.set_xlabel("granularity N"); ax.set_ylabel("A_final (median over seeds)")
    ax.set_title("(b) final damage vs N"); ax.legend(fontsize=7)
    savefig(fig, "fig_b_damage_vs_N.png")

def fig_c():
    fig, ax = plt.subplots(1, 1, figsize=(6, 4.5), squeeze=False); ax = ax[0, 0]
    seen = set()
    for c in CONDS.values():
        if c.get("M") != M_TASKS or c.get("status") != "done":
            continue
        t1 = [t for t in tasks1(c) if t.get("A") is not None and t["D_K_cum"] > 0]
        lab = c["learner"] if c["learner"] not in seen else None
        seen.add(c["learner"])
        mk = {"tanh": "s", "linear": "x"}.get(c["model"], "o" if c["seq"] == "R" else "^")
        ax.scatter([t["D_K_cum"] for t in t1], [t["A"] for t in t1], s=10, color=COLORS.get(c["learner"], "k"),
                   alpha=0.6, label=lab, marker=mk)
    ax.set_xscale("log"); ax.set_xlabel("cumulative dose D_cum (tasks 1..m)"); ax.set_ylabel("damage A")
    ax.set_title("(c) collapse, all conditions (o ReLU-R, ^ ReLU-E, s tanh, x linear)"); ax.legend(fontsize=7)
    savefig(fig, "fig_c_collapse.png")

def fig_d():
    fig, axs = plt.subplots(1, 5, figsize=(17, 3.6), squeeze=False)
    for i, (lrn, grid) in enumerate(FIVE):
        ax = axs[0, i]
        c = getc(grid, "relu", "R", lrn, 1, SEEDS[0])
        if c is not None:
            for t in c["tasks"]:
                if "filter" in t:
                    lam = np.array(t["filter"]["lam_ref"], float)
                    sj = np.array(t["filter"]["S_jump"], float); sr = np.array(t["filter"]["S_res"], float)
                    msk = (lam > EIG_FLOOR * lam[0]) & (sj > 0) & (sr > 0)
                    ax.loglog(lam[msk], sj[msk] / sr[msk], ".", ms=3, alpha=0.6, label=f"task {t['m']} (alpha={f4(t['alpha'])})")
            ax.legend(fontsize=6)
        ax.set_title(f"(d) {lrn}: S_jump/S_res"); ax.set_xlabel("lambda_i of K_ref")
    savefig(fig, "fig_d_filter.png")

def fig_e():
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), squeeze=False); ax = ax[0, 0]
    names, shares = [], []
    for lrn, grid in FIVE + [("GD-half", "G-lr"), ("GD-quarter", "G-lr")]:
        c = getc(grid, "relu", "R", lrn, 1, SEEDS[0])
        if c is None:
            continue
        tb = [sum(t["D_band"][b] for t in tasks1(c)) for b in ("TOP", "MID", "BOT")]
        tot = sum(tb)
        if tot > 0:
            names.append(lrn); shares.append([v / tot for v in tb])
    if shares:
        sh = np.array(shares); bottom = np.zeros(len(names))
        for j, (b, col) in enumerate(zip(("TOP", "MID", "BOT"), ("#4c72b0", "#dd8452", "#c44e52"))):
            ax.bar(names, sh[:, j], bottom=bottom, color=col, label=b); bottom += sh[:, j]
    ax.set_ylabel("share of cumulative D_K"); ax.set_title("(e) where the dose falls (N = 1)"); ax.legend(fontsize=7)
    savefig(fig, "fig_e_band_shares.png")

def fig_f():
    fig, ax = plt.subplots(1, 1, figsize=(6, 4), squeeze=False); ax = ax[0, 0]
    seen = set()
    for r in pooled_records({"G-main", "G-gn"}, SEEDS):
        if r.get("psi") is not None and r["psi"] > 0:
            ax.scatter(r["A"], r["psi"], s=10, color=COLORS[r["learner"]], alpha=0.6,
                       label=r["learner"] if r["learner"] not in seen else None)
            seen.add(r["learner"])
    ax.set_yscale("log"); ax.set_xlabel("damage A"); ax.set_ylabel("probe psi = L_300/L_0")
    ax.set_title("(f) probe trainability vs damage"); ax.legend(fontsize=7)
    savefig(fig, "fig_f_psi_vs_A.png")

def fig_g():
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), squeeze=False)
    for lrn in ["GN", "GD", "ADAM"]:
        e = grid_eps("G-eps")
        D = [med([S(getc("G-eps", "relu", "E", lrn, 1, s, eps=x), "mean_task_DK") for s in SEEDS]) for x in e]
        A = [med([S(getc("G-eps", "relu", "E", lrn, 1, s, eps=x), "A_final") for s in SEEDS]) for x in e]
        pD = [(x, y) for x, y in zip(e, D) if y is not None and y > 0]
        pA = [(x, y) for x, y in zip(e, A) if y is not None and y > 0]
        if pD: axs[0, 0].loglog([p[0] for p in pD], [p[1] for p in pD], "-o", color=COLORS[lrn], label=lrn)
        if pA: axs[0, 1].loglog([p[0] for p in pA], [p[1] for p in pA], "-o", color=COLORS[lrn], label=lrn)
    axs[0, 0].set_title("(g) mean per-task dose vs eps"); axs[0, 1].set_title("(g) A_final vs eps (positive values only)")
    for ax in axs[0]:
        ax.set_xlabel("eps"); ax.legend(fontsize=7)
    savefig(fig, "fig_g_eps.png")

if HAVE_MPL:
    os.makedirs(FIG_DIR, exist_ok=True)
    for _fn in [fig_a, fig_b, fig_c, fig_d, fig_e, fig_f, fig_g]:
        try:
            _fn()
        except Exception as _e:
            print(f"figure {_fn.__name__} failed: {_e!r}")
            try:
                plt.close("all")
            except Exception:
                pass
    print("figures saved:", FIG_FILES)
else:
    print("matplotlib is not available: figures skipped.")

## 17. Summary and download
Prints a compact summary and zips `results_part5.json` and the figures into `part5_outputs.zip`.

In [ ]:
RES = load_results()
merge_condition_files(RES, RUN_DIR)
_done = [c for c in RES["conditions"].values() if c.get("M") == M_TASKS]
print(f"MODE={MODE} | data_source={part5lib.DATA_SOURCE} | device={DEVICE} | seeds={SEEDS} | M={M_TASKS} | n={N_TRAIN}")
print(f"conditions in results: {len(_done)} of {len(all_conditions())} planned "
      f"({sum(c.get('status') == 'done' for c in _done)} done, {sum(c.get('status') != 'done' for c in _done)} not done)")
print("learning rates: " + "; ".join(f"{k}: " + ", ".join(f"{o}={d['chosen']:g}" for o, d in v.items())
                                     for k, v in LR_SEL.items()))
print("verdicts" + (" (PRELIMINARY)" if PRELIM else "") + ":")
for h, v in VERDICTS.items():
    print(f"  {h}: {v}")
with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RESULTS_FILE)
    for p in sorted(glob.glob(os.path.join(FIG_DIR, "*.png"))):
        z.write(p)
print(f"wrote {ZIP_FILE} ({os.path.getsize(ZIP_FILE) / 1e6:.2f} MB)")
mirror(ZIP_FILE)
print(f"results: {os.path.abspath(RESULTS_FILE)}")
print(f"zip:     {os.path.abspath(ZIP_FILE)}")
if IN_COLAB:
    try:
        from google.colab import files
        files.download(ZIP_FILE)
    except Exception as _e:
        print("download failed (use the Files panel on the left):", _e)
else:
    print("Not on Colab: download part5_outputs.zip (and results_part5.json) from the Jupyter file browser "
          "(right-click -> Download).")